# VCC 2026 · 3c — the profile metrics: what calling costs, and what the generator allows

Three of the six leaderboard metrics are about which genes we call (fidelity, reach, Jaccard), and `vcc_2026_3b` showed that choice has run out of road on H1. The other points are in the **profile metrics**: pds (can the knockdowns be told apart) and mse (do the cells look like the real ones). mse has been 0 for every arm and submission. This notebook asks why, in three steps:
- **0a. The finalists' profile scores** (from `vcc_2026_3`'s `val_final`, nothing rerun): what calling costs on pds and mse — non-gated against top-N against the gate.
- **A. "Honest profile" arms:** Model 2 with every gene at its own Δ — no top-N, no detection floor — at α ∈ {0.5, 0.75, 1}. Rationale: the leaderboard's top entries reportedly sit near the baseline on the DE metrics (raw fidelity ≈ 0.53, Jaccard ≈ 0.037) and win on mse and pds. Risk: without the shared response the DE metrics collapse (v1: fidelity −1.69), so a leaderboard test would pair it with the generic-response probe.
- **0b. The H1 oracle on the leaderboard scale:** H1's own measured effects (from half of each knockdown's cells) pushed through our generator and scored against the other half, with its own generic baseline and replicate anchor. If even the oracle scores mse 0, **the generator is the bottleneck, not the effect model.**

Code: `vcc_2026_3b`'s scripts; the generator's oracle mode can now score several arms (`oracle` / `nochange`, any α and cell rule) and build its own bundle (`--scale`). Item A is scored against `vcc_2026_3`'s finalists bundle, so it compares directly with every earlier arm.

## 0. Session
Paths, the `run` / `bg` / `status` helpers, and input checks.

In [ ]:
import os, sys, shutil, subprocess, shlex, json
from pathlib import Path
import numpy as np
import pandas as pd
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V / 'code' / 'vcc_2026_3c'), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_3c'),
           B='gs://vcc2026', PY=sys.executable, WORK=str(V / 'lib' / 'vcc_2026_3c'), WORK3=str(V / 'lib' / 'vcc_2026_3'),
           WORK3B=str(V / 'lib' / 'vcc_2026_3b'), H1_CELLS=str(V / 'lib/h1/H1_full_2025.h5ad'))
os.environ.update(ENV)
for d in ('CODE', 'LOGS', 'WORK'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now (from the code folder), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/vcc_2026_3c/<name>.log."""
    session = 'vcc3c_' + name
    if subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0:
        raise RuntimeError(f'{session} is already running; wait for completion')
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    session = 'vcc3c_' + name
    alive = subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else ('NOT STARTED' if not lines else 'STOPPED - read the log below'))
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

# ---- checks ----
W3, W3B = Path(ENV['WORK3']), Path(ENV['WORK3B'])
need = [W3 / 'validation/targets.csv', W3 / 'validation/pred/H1/re.npy', W3 / 'val_final/benchmark.json', W3 / 'val_final/bundle',
        W3 / 'val_final/metric_ranks.csv', W3B / 'val_3b/scaled_scores.csv', Path(ENV['H1_CELLS'])]
missing = [str(f) for f in need if not f.exists()]
print('python:', sys.executable)
print('inputs: all present' if not missing else f'MISSING inputs: {missing} (run vcc_2026_3 and vcc_2026_3b first)')
print('official evaluator:', shutil.which('cell-eval2') or 'MISSING: pip install -r requirements-eval.txt')

## 1. Scripts
`vcc_2026_3b`'s five scripts; only `vcc3_generate.py` changes: oracle mode accepts `--variants` (methods `oracle` and `nochange`) and `--scale` (build the competition bundle on the oracle's real cells and score every arm with the leaderboard formula). Run all five cells after editing; they write to `~/vcc/code/vcc_2026_3c/`.

### vcc2_common.py
Shared helpers, unchanged from `vcc_2026_2`.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_3c/vcc2_common.py
"""Shared helpers for the VCC 2026 meta-analysis pipeline.

Imported by:
  vcc2_meta.py      : read_names, save_json
  vcc2_generate.py  : seed_for, save_json, expression_gate, int64_indptr, validate_counts, alignment, moments
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

CONTROL = "non-targeting"


def read_names(path, col):
    """One column of a CSV as a numpy string array (first column if `col` is absent)."""
    df = pd.read_csv(path)
    return np.asarray(df[col] if col in df.columns else df.iloc[:, 0], dtype=str)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=1, default=_json_default))


def seed_for(seed, *keys):
    """Deterministic seed from (seed, keys...): stable across runs and machines (unlike Python's hash())."""
    h = hashlib.sha256(repr((int(seed),) + tuple(str(k) for k in keys)).encode()).digest()
    return int.from_bytes(h[:8], "little")


def expression_gate(mean_cpm):
    """Strict scorer boundary, using arithmetic mean per-cell CPM in real controls."""
    mean = np.asarray(mean_cpm, dtype=float)
    return np.isfinite(mean) & (mean > 5.0)


def int64_indptr(matrix):
    """CSR with an int64 row pointer.

    An h5ad X that is written once and then grown with append() keeps the dtype of its
    FIRST block's indptr. scipy uses int32 for small blocks, which caps the finished file at
    2**31 stored values (merged H1 has ~3.5e9; a 360k-cell submission is ~2.2e9).
    """
    m = matrix.tocsr()
    m.indptr = m.indptr.astype(np.int64)
    return m


def validate_counts(matrix, label="counts"):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix)
    if not (np.isfinite(values).all() and (values >= 0).all() and (values == np.floor(values)).all()):
        raise ValueError(f"{label}: expected finite, nonnegative integer raw counts")


def alignment(source_names, genes):
    """For each of OUR genes, its column in the source (-1 = the source did not measure it)."""
    pos = {g: i for i, g in enumerate(np.asarray(source_names, dtype=str))}
    idx = np.array([pos.get(g, -1) for g in np.asarray(genes, dtype=str)], dtype=np.int64)
    if (idx >= 0).sum() == 0:
        raise ValueError("no genes in common with the provided gene list")
    return idx


def _matrix(a, layer):
    if layer is None:
        return a.X
    if layer in a.layers:
        return a.layers[layer]
    raise KeyError(f"layer {layer!r} not found")


def moments(a, rows, idx, layer=None, block=512):
    """Per-gene statistics over `rows` (cells), streamed `block` cells at a time. Library size = sum over OUR genes.
    Returns arrays aligned to our gene list (NaN where the source lacks the gene)."""
    X = _matrix(a, layer); G = len(idx); ok = idx >= 0; cols = idx[ok]
    s_cnt = np.zeros(ok.sum()); s_cpm = np.zeros(ok.sum()); s_cpm2 = np.zeros(ok.sum()); s_det = np.zeros(ok.sum())
    libs = []
    for b in range(0, len(rows), block):
        r = np.asarray(rows[b:b + block])
        M = X[r]
        M = (M.tocsr() if sp.issparse(M) else sp.csr_matrix(np.asarray(M)))[:, cols].astype(np.float64)
        validate_counts(M)
        lib = np.asarray(M.sum(1)).ravel(); libs.append(lib)
        C = sp.diags(1e6 / np.maximum(lib, 1)) @ M
        s_cnt += np.asarray(M.sum(0)).ravel(); s_cpm += np.asarray(C.sum(0)).ravel()
        s_cpm2 += np.asarray(C.multiply(C).sum(0)).ravel(); s_det += np.asarray((M > 0).sum(0)).ravel()
    n = len(rows); lib = np.concatenate(libs) if libs else np.array([0.0])

    def full(v):
        out = np.full(G, np.nan, dtype=np.float32); out[ok] = v; return out
    if n == 0:
        raise ValueError("no cells to profile")
    mean = s_cpm / n; var = np.maximum(s_cpm2 / n - mean ** 2, 0)
    pooled = s_cnt / max(s_cnt.sum(), 1) * 1e6
    stats = dict(mean_cpm=np.where(ok, 0.0, np.nan), pooled_cpm=full(pooled), bulk_cpm=full(pooled), log_mean=full(np.log1p(pooled)),
                 detection=full(s_det / n), var_cpm=full(var), se_mean=full(np.sqrt(var / n)),
                 n_cells=np.int64(n), median_library=np.float64(np.median(lib)), measured=ok.copy(),
                 legacy_baseline=np.bool_(False))
    stats["mean_cpm"][ok] = mean  # float64 preserves the strict 5-CPM boundary
    return stats

### vcc2_meta.py
Model 2 (precision-weighted average), as in `vcc_2026_2` but voting over all eight lines: the baseline for variant (b) and a reference arm.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_3c/vcc2_meta.py
"""Random-effects meta-analysis of the public panel sources: precision-weighted average (Model 2).

Inputs: lib/sources/<NAME>/effects_panel.h5ad from scripts/build_panel_sources.py (575 targets x 18,533 genes,
layers lfc / se, NaN = not available). One source per background: the T cells enter as CD4_Stim48hr only.

  fit      per-gene tau2: DerSimonian-Laird, pooled over every target with >= 2 backgrounds -> model dir
  predict  one delta table per line, in the folder format vcc2_generate.py reads:
             re.npy, re_se.npy         weight = 1/(SE^2 + tau2); prediction = sum(weight x lfc) / sum(weight);
                                       SE = sqrt(1 / sum(weight))
             re_z.npy                  |re| / re_se: how convincing each prediction is (ranks genes for top-N calling)
             n_bg.npy                  backgrounds that measured the gene (float; NaN where none did)
             sign_agree.npy            share of those backgrounds whose lfc has the sign of the weighted mean
             n_backgrounds.npy         backgrounds behind each value; 0 = no information (delta stored as NaN)
             ontarget_lfc.npy          mean of the sources' ontarget_lfc: the knocked-down gene, applied separately
  exam     the held-out background's kept targets that at least one other background also kept
  explain  one target x gene: each background's lfc, SE, weight and share, then the prediction

The knocked-down gene's own column never enters tau2 or the averages. No shrinkage: the average is the prediction.

Example: python vcc2_meta.py fit --sources-dir ~/vcc/lib/sources --genes gene_names.csv --holdout H1 --out models/exam
         python vcc2_meta.py predict --model models/exam --targets exam_targets.csv --lines H1 --out exam
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from vcc2_common import read_names, save_json

SCHEMA = 'vcc_2026_2_meta_v1'
SOURCES = ['HCT116', 'HEK293T', 'K562_gwps', 'CD4_Stim48hr', 'H1', 'RPE1', 'HepG2', 'Jurkat']   # step 1: Stim48hr is the one T-cell vote
MODELS = ('re',)


def background_of(root, name):
    a = ad.read_h5ad(Path(root) / name / 'effects_panel.h5ad', backed='r')
    try: return str(a.uns['background']), a.obs.copy()
    finally: a.file.close()


def included(root, names, holdout=None):
    """Sources that vote, in order; refuses two sources from one background."""
    chosen, seen = [], {}
    for name in names:
        background, _ = background_of(root, name)
        if background == holdout: continue
        if background in seen: raise ValueError(f'{name} and {seen[background]} are both {background}: one vote per background')
        seen[background] = name; chosen.append((name, background))
    if not chosen: raise ValueError('no source backgrounds remain')
    return chosen


class Panel:
    """Included sources stacked as sources x targets x genes (lfc, se), plus each source's on-target LFC."""
    def __init__(self, root, sources, genes):
        self.names = [n for n, _ in sources]; self.backgrounds = [b for _, b in sources]
        lfc, se, own = [], [], []
        for name in self.names:
            a = ad.read_h5ad(Path(root) / name / 'effects_panel.h5ad')
            if not np.array_equal(a.var_names.astype(str), genes): raise ValueError(f'{name}: gene order differs from gene_names.csv')
            targets = a.obs_names.astype(str).to_numpy()
            if lfc and not np.array_equal(targets, self.targets): raise ValueError(f'{name}: target order differs between panels')
            self.targets = targets
            kept = (a.obs['status'] == 'kept').to_numpy()
            lfc.append(np.asarray(a.layers['lfc'], np.float32)); se.append(np.asarray(a.layers['se'], np.float32))
            own.append(np.where(kept, a.obs['ontarget_lfc'].to_numpy(float), np.nan))
        self.lfc, self.se, self.ontarget = np.stack(lfc), np.stack(se), np.stack(own)
        gp = {g: i for i, g in enumerate(genes)}
        self.own_col = np.array([gp.get(t, -1) for t in self.targets])
        self.row = {t: i for i, t in enumerate(self.targets)}


def tau2_dl(lfc, se, own_col):
    """Per-gene between-background variance, DerSimonian-Laird pooled over targets.

    For each target with k >= 2 backgrounds: Q = sum w (y - mu_FE)^2 with w = 1/SE^2 (noise alone predicts k - 1)
    and C = sum w - sum w^2 / sum w.  tau2 = max(0, (sum Q - sum (k - 1)) / sum C), floored at 0.
    Returns tau2 (NaN where no target ever had two backgrounds) and the pooled degrees of freedom.
    """
    _, T, G = lfc.shape
    Q, df, C = np.zeros(G), np.zeros(G), np.zeros(G)
    for t in range(T):
        y, v = lfc[:, t].astype(np.float64), se[:, t].astype(np.float64) ** 2
        ok = np.isfinite(y) & np.isfinite(v) & (v > 0)
        if own_col[t] >= 0: ok[:, own_col[t]] = False
        k = ok.sum(0); use = k >= 2
        if not use.any(): continue
        w = np.where(ok, 1 / np.where(ok, v, 1), 0); y = np.where(ok, y, 0)
        sw = w.sum(0); mu = np.divide((w * y).sum(0), sw, out=np.zeros(G), where=sw > 0)
        Q[use] += (w * (y - mu) ** 2).sum(0)[use]; df[use] += k[use] - 1
        C[use] += (sw - np.divide((w ** 2).sum(0), sw, out=np.zeros(G), where=sw > 0))[use]
    tau2 = np.full(G, np.nan); est = C > 0
    tau2[est] = np.maximum(0, (Q[est] - df[est]) / C[est])
    return tau2, df


def combine(lfc, se, tau2):
    """Sources x rows x genes -> (re, re_se, k, sign_agree) per row x gene. k == 0 gives NaN (no information).
    sign_agree is the share of the k backgrounds whose lfc has the same sign as the weighted mean."""
    y = lfc.astype(np.float64); total = se.astype(np.float64) ** 2 + tau2   # SE^2 + tau2
    ok = np.isfinite(y) & np.isfinite(total)
    k = ok.sum(0); y = np.where(ok, y, 0)
    w = np.where(ok, 1 / np.where(ok, total, 1), 0); sw = w.sum(0)
    with np.errstate(divide='ignore', invalid='ignore'):
        mean = (w * y).sum(0) / sw
        agree = (ok & (np.sign(y) == np.sign(mean))).sum(0) / k
        out = (mean, np.sqrt(1 / sw))
    re, re_se = (np.where(k > 0, x, np.nan).astype(np.float32) for x in out)
    return re, re_se, k.astype(np.uint8), np.where(k > 0, agree, np.nan).astype(np.float32)


def fit(sources_dir, genes_csv, output, holdout=None, names=SOURCES):
    genes = read_names(genes_csv, 'gene_name'); sources = included(sources_dir, names, holdout)
    panel = Panel(sources_dir, sources, genes)
    tau2, df = tau2_dl(panel.lfc, panel.se, panel.own_col)
    est = np.isfinite(tau2); fallback = float(np.median(tau2[est]))
    measured_in = np.isfinite(panel.lfc).any(1).sum(0)                  # backgrounds measuring each gene at all
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    np.save(out / 'genes.npy', genes); np.save(out / 'tau2.npy', tau2.astype(np.float32)); np.save(out / 'tau2_df.npy', df)
    q = np.quantile(tau2[est], [0.1, 0.25, 0.5, 0.75, 0.9])
    model = dict(schema=SCHEMA, sources_dir=str(Path(sources_dir).resolve()), genes_csv=str(Path(genes_csv).resolve()),
                 sources=[n for n, _ in sources], backgrounds=[b for _, b in sources], heldout_background=holdout,
                 tau2_estimator='DerSimonian-Laird, pooled over targets with >= 2 backgrounds, own column excluded, floored at 0',
                 tau2_fallback=fallback, tau2_fallback_rule='median tau2 for genes no target measured in 2 backgrounds',
                 genes_estimated=int(est.sum()), genes_fallback=int((~est & (measured_in > 0)).sum()),
                 genes_no_data=int((measured_in == 0).sum()), genes_tau2_zero=int((tau2[est] == 0).sum()),
                 tau2_quantiles=dict(zip(['10%', '25%', '50%', '75%', '90%'], q.round(5).tolist())))
    save_json(out / 'model.json', model)
    print(f'fit: {len(sources)} backgrounds {model["backgrounds"]} (held out: {holdout})')
    print(f'  tau2 estimated for {model["genes_estimated"]:,} genes ({model["genes_tau2_zero"]:,} at 0), '
          f'median {fallback:.4f}; {model["genes_fallback"]:,} genes measured in one background use the median; '
          f'{model["genes_no_data"]:,} genes have no data anywhere')
    return model


def load_model(model_dir):
    m = Path(model_dir); model = json.loads((m / 'model.json').read_text())
    if model.get('schema') != SCHEMA: raise ValueError('incompatible model; fit with this notebook')
    genes = np.load(m / 'genes.npy').astype(str); tau2 = np.load(m / 'tau2.npy').astype(np.float64)
    used = np.where(np.isfinite(tau2), tau2, model['tau2_fallback'])
    panel = Panel(model['sources_dir'], list(zip(model['sources'], model['backgrounds'])), genes)
    return model, genes, used, panel


def predict(model_dir, targets_csv, lines, output, chunk=64):
    model, genes, tau2, panel = load_model(model_dir)
    targets = read_names(targets_csv, 'target_gene')
    if not len(targets): raise ValueError('empty perturbation panel')
    out = Path(output)
    for line in lines:   # an earlier meta prediction is replaced (it is deterministic); anything else is kept
        d = out / line; earlier = d / 'prediction_report.json'
        if d.exists() and any(d.iterdir()) and not (earlier.exists() and json.loads(earlier.read_text()).get('schema') == SCHEMA):
            raise ValueError(f'choose an empty prediction directory: {d}')
        if d.exists():
            for old in [*d.glob('*.npy'), d / 'prediction_report.json']: old.unlink(missing_ok=True)
    T, G = len(targets), len(genes)
    arrays = {name: np.full((T, G), np.nan, np.float32) for name in ['re', 're_se', 'sign_agree']}
    count = np.zeros((T, G), np.uint8); ontarget = np.full(T, np.nan, np.float32)
    rows = np.array([panel.row.get(t, -1) for t in targets]); missing = targets[rows < 0].tolist()
    gp = {g: i for i, g in enumerate(genes)}
    for start in range(0, T, chunk):
        idx = np.arange(start, min(start + chunk, T)); idx = idx[rows[idx] >= 0]
        if not len(idx): continue
        lfc, se = panel.lfc[:, rows[idx]].copy(), panel.se[:, rows[idx]]
        for j, i in enumerate(idx):
            if targets[i] in gp: lfc[:, j, gp[targets[i]]] = np.nan   # step 5: own gene is not averaged
        arrays['re'][idx], arrays['re_se'][idx], k, arrays['sign_agree'][idx] = combine(lfc, se, tau2)
        count[idx] = k
        own = panel.ontarget[:, rows[idx]]; n = np.isfinite(own).sum(0)
        ontarget[idx] = np.divide(np.nan_to_num(own).sum(0), n, out=np.full(len(idx), np.nan), where=n > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        arrays['re_z'] = (np.abs(arrays['re']) / arrays['re_se']).astype(np.float32)   # NaN where re is NaN
    arrays['n_bg'] = np.where(count > 0, count, np.nan).astype(np.float32)
    voters = [[b for b, kept in zip(panel.backgrounds, np.isfinite(panel.ontarget[:, r])) if kept] if r >= 0 else [] for r in rows]
    info = (count > 0).sum(1)
    report = dict(schema=SCHEMA, models=list(MODELS), heldout_background=model['heldout_background'],
                  source_backgrounds=model['backgrounds'], source_names=model['sources'], model_dir=str(Path(model_dir).resolve()),
                  targets=str(Path(targets_csv).resolve()), weighting='1 / (SE^2 + tau2)', shrinkage='none', ontarget='mean of the backgrounds\' ontarget_lfc, kept rows only',
                  no_information='NaN in re (n_backgrounds 0); the generator predicts no change there',
                  not_in_panel=missing, no_source_targets=[t for t, v in zip(targets.tolist(), voters) if not v],
                  coverage=[dict(target=t, backgrounds=v, genes_with_information=int(n)) for t, v, n in zip(targets.tolist(), voters, info)])
    for line in lines:
        d = out / line; d.mkdir(parents=True, exist_ok=True)
        np.save(d / 'genes.npy', genes); np.save(d / 'targets.npy', targets)
        for name, value in arrays.items(): np.save(d / f'{name}.npy', value)
        np.save(d / 'n_backgrounds.npy', count); np.save(d / 'ontarget_lfc.npy', ontarget)
        save_json(d / 'prediction_report.json', dict(report, line=line))
    by_k = pd.Series([len(v) for v in voters]).value_counts().sort_index()
    print(f'predict: {T} targets x {G:,} genes -> {", ".join(str(out / l) for l in lines)}')
    print('  targets by number of backgrounds: ' + ', '.join(f'{k}: {n}' for k, n in by_k.items()))
    print(f'  no source at all (all genes "no information"): {report["no_source_targets"] or "none"}')
    if missing: print(f'  not in the panel: {missing}')
    return report


def exam(sources_dir, holdout, output, names=SOURCES):
    status = {}
    for name in names:
        background, obs = background_of(sources_dir, name)
        status.setdefault(background, []).append(obs['status'].eq('kept'))
    if holdout not in status: raise ValueError(f'no source for {holdout}')
    held = pd.concat(status.pop(holdout), axis=1).any(axis=1)
    others = pd.concat([pd.concat(v, axis=1).any(axis=1) for v in status.values()], axis=1).sum(axis=1)
    targets = held.index[held & (others >= 1)].tolist()
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'target_gene': targets}).to_csv(output, index=False)
    print(f'exam: {len(targets)} {holdout} targets also kept in >= 1 other background -> {output}')
    return targets


def explain(model_dir, target, gene):
    """The worked example for one target x gene, from the real panel."""
    if target == gene: raise ValueError('the knocked-down gene is not averaged; see ontarget_lfc')
    model, genes, tau2, panel = load_model(model_dir)
    r, g = panel.row[target], int(np.flatnonzero(genes == gene)[0])
    t2 = tau2[g]; y = panel.lfc[:, r, g].astype(float); se = panel.se[:, r, g].astype(float)
    frame = pd.DataFrame(dict(background=panel.backgrounds, lfc=y, se=se)).set_index('background')
    frame['se2_plus_tau2'] = se ** 2 + t2; frame['weight'] = 1 / frame.se2_plus_tau2
    frame['share'] = frame.weight / frame.weight.sum()
    re, re_se, k, agree = (x[0, 0] for x in combine(frame.lfc.to_numpy()[:, None, None],
                                                                 frame.se.to_numpy()[:, None, None], np.array([t2])))
    summary = dict(target=target, gene=gene, tau2=float(t2), tau2_estimated=bool(np.isfinite(np.load(Path(model_dir) / 'tau2.npy')[g])),
                   n_backgrounds=int(k), re=float(re), re_se=float(re_se), re_z=float(abs(re) / re_se), sign_agree=float(agree))
    return frame, summary


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)
    f = sub.add_parser('fit')
    for k in ['sources-dir', 'genes', 'out']: f.add_argument('--' + k, required=True)
    f.add_argument('--holdout'); f.add_argument('--sources', nargs='+', default=SOURCES)
    pr = sub.add_parser('predict')
    for k in ['model', 'targets', 'out']: pr.add_argument('--' + k, required=True)
    pr.add_argument('--lines', nargs='+', required=True); pr.add_argument('--chunk', type=int, default=64)
    e = sub.add_parser('exam')
    for k in ['sources-dir', 'holdout', 'out']: e.add_argument('--' + k, required=True)
    e.add_argument('--sources', nargs='+', default=SOURCES)
    x = sub.add_parser('explain')
    for k in ['model', 'target', 'gene']: x.add_argument('--' + k, required=True)
    a = p.parse_args()
    if a.cmd == 'fit': fit(a.sources_dir, a.genes, a.out, a.holdout, a.sources)
    elif a.cmd == 'predict': predict(a.model, a.targets, a.lines, a.out, a.chunk)
    elif a.cmd == 'exam': exam(a.sources_dir, a.holdout, a.out, a.sources)
    else:
        frame, summary = explain(a.model, a.target, a.gene)
        print(frame.to_string(float_format='%.4f')); print(json.dumps(summary, indent=2))

### vcc3_ridge.py
Programs (PCA), program scores, rotating-destination ridge for both variants, grouped CV, and the Δ tables.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_3c/vcc3_ridge.py
"""PCA programs + ridge transfer, on top of the Model 2 baseline (vcc_2026_3).

Inputs
  lib/filters/<NAME>.h5ad  every QC-passed knockdown of each source (X = lfc, layers/se, var ctrl_cpm): training rows
  lib/sources panels       only to refit each training destination's Model 2 tau2 without that destination
  a Model 2 prediction dir (vcc2_meta.py predict) for the line(s) being predicted, built with the same holdout

fit
  genes     one fixed gene set: measured (>= 1 CPM) in HCT116, HEK293T and K562, in gene_names.csv order
  programs  uncentred PCA of every knockdown of the program sources (Orion, K562, RPE1, HepG2, Jurkat) on that set.
            Each row's own knocked-down gene is set to 0 (its drop is handled separately by ontarget_lfc), so every
            row uses the same columns. The newer lines miss 6-10% of the set; those entries are filled iteratively
            from the low-rank fit (EM-PCA) instead of shrinking the gene set to what every line measured
  scores    every source row -> k numbers by least squares on the genes that source measured
  pairs     the destination d rotates over every included line; inputs are the other lines' scores for the same
            knockdown (0 + missing flag where a line did not keep it; d's own slot is always missing):
              (a) programs   target = d's program scores
              (b) corrected  target = scores of d's residual  R = measured d - Model 2 built without d
                             (tau2 refit without d); Model 2's own scores are an extra input
  ridge     one small ridge per program and variant; penalty from a fixed grid by 5-fold CV grouped by knockdown
            (all pairs of a knockdown share a fold): this measures transfer to unseen knockdowns in known lines,
            not to unseen lines. The H1 run in the notebook is validation, not an untouched exam.

predict
  (a) programs_k<k>  = loadings x predicted scores on the program genes; NaN elsewhere (no prediction)
  (b) corrected_k<k> = Model 2 + loadings x predicted residual scores on the program genes; Model 2 elsewhere
  Rows with all slots present (never seen in training, where the destination's slot is always missing) are
  predicted with each slot masked in turn, then averaged. For corrected, each mask also rebuilds Model 2 without
  that slot (tau2 refit too), exactly as training built its baseline, and the output averages
  (Model 2 without the slot + correction). Other rows are unchanged.
  <variant>_k<k>_spread.npy stores the program-space SD across masks (NaN on unaffected rows, not an SE).

Choice of k: besides the fixed grid, k80 = the smallest k whose programs hold 80% of the variance ABOVE THE NOISE.
  The noise spectrum comes from a table of pure measurement noise (each entry drawn with its own SE), so the
  comparison respects that some genes are far noisier than others; components above it carry signal.
  The knocked-down gene's own column is NaN (ontarget_lfc from Model 2 applies), and targets that no line kept stay
  NaN ("no information"). No SE is written for the ridge outputs: the generator's SE gate is off for them (z* = 0).
  Nothing about the destination enters, so the Delta tables for A, B and C are identical.

Example: python vcc3_ridge.py fit --filters-dir ~/vcc/lib/filters --sources-dir ~/vcc/lib/sources \
             --genes gene_names.csv --holdout H1 --out models/validation
         python vcc3_ridge.py predict --model models/validation --baseline validation/model2/H1 --lines H1 --out validation/pred
"""
from __future__ import annotations

import argparse
import json
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

try:
    from anndata.io import read_elem
except ImportError:  # older anndata
    from anndata.experimental import read_elem

import vcc2_meta as meta
from vcc2_common import read_names, save_json, seed_for

SCHEMA = 'vcc_2026_3_ridge_v1'
BACKGROUND = dict(HCT116='HCT116', HEK293T='HEK293T', K562_gwps='K562', CD4_Stim48hr='CD4_T', H1='H1',
                  RPE1='RPE1', HepG2='HepG2', Jurkat='Jurkat')      # Replogle RPE1; Nadig HepG2 / Jurkat
LINES = list(BACKGROUND)                                   # one source per background: T cells = Stim48hr only
GENE_SET_SOURCES = ['HCT116', 'HEK293T', 'K562_gwps']      # the program genes: measured in all three
PROGRAM_SOURCES = ['HCT116', 'HEK293T', 'K562_gwps', 'RPE1', 'HepG2', 'Jurkat']   # every non-H1 cell line learns programs
K_GRID = (5, 10, 20)                                       # 20 is the ceiling: 21 components clear the noise edge
PENALTIES = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)                  # fixed grid, chosen per program by grouped CV
SIGNAL_SHARE = 0.8                                         # k80: share of the above-noise variance the programs keep
FOLDS = 5
MIN_CPM = 1.0                                              # same gene filter as the panel tables
SEED = 2026


# ---------------------------------------------------------------- data
def check_backgrounds(sources_dir):
    for name in LINES:
        b, _ = meta.background_of(sources_dir, name)
        if b != BACKGROUND[name]:
            raise ValueError(f'{name}: panel says {b!r}, vcc3 says {BACKGROUND[name]!r}')


def var_of(filters_dir, name):
    with h5py.File(Path(filters_dir) / f'{name}.h5ad', 'r') as f:
        return read_elem(f['var']), read_elem(f['obs'])


def program_genes(filters_dir, genes, names=GENE_SET_SOURCES):
    """Challenge genes measured (>= MIN_CPM in controls) in every program source, in gene list order."""
    keep = np.ones(len(genes), bool)
    for name in names:
        var, _ = var_of(filters_dir, name)
        cpm = dict(zip(var.index.astype(str), var['ctrl_cpm'].to_numpy(float)))
        keep &= np.array([cpm.get(g, np.nan) >= MIN_CPM for g in genes])
    return np.asarray(genes)[keep]


class Line:
    """One source's QC-passed knockdowns on the program genes: lfc and se (NaN = not measured there)."""
    def __init__(self, filters_dir, name, genes, block=512, with_se=True):
        self.name, self.background = name, BACKGROUND[name]
        var, obs = var_of(filters_dir, name)
        self.targets = obs.index.astype(str).to_numpy()
        pos = {g: i for i, g in enumerate(var.index.astype(str))}
        cols = np.array([pos.get(g, -1) for g in genes]); cpm = np.full(len(genes), np.nan)
        ok = cols >= 0; cpm[ok] = var['ctrl_cpm'].to_numpy(float)[cols[ok]]
        self.measured = ok & (cpm >= MIN_CPM)
        src, dst = cols[self.measured], np.flatnonzero(self.measured)
        shape = (len(self.targets), len(genes))
        self.lfc = np.full(shape, np.nan, np.float32); self.se = np.full(shape, np.nan, np.float32) if with_se else None
        with h5py.File(Path(filters_dir) / f'{name}.h5ad', 'r') as f:   # row blocks: never the whole matrix at once
            for start in range(0, shape[0], block):
                stop = min(start + block, shape[0])
                self.lfc[start:stop, dst] = f['X'][start:stop][:, src]
                if with_se: self.se[start:stop, dst] = f['layers/se'][start:stop][:, src]
        gp = {g: i for i, g in enumerate(genes)}
        self.own = np.array([gp.get(t, -1) for t in self.targets])
        self.row = {t: i for i, t in enumerate(self.targets)}


def own_set(values, own, fill):
    """Copy with each row's own knocked-down gene set to `fill` (0 for programs/scores, NaN for averaging)."""
    out = np.array(values, np.float32, copy=True); r = np.flatnonzero(own >= 0)
    out[r, own[r]] = fill
    return out


# ---------------------------------------------------------------- programs and scores
def fit_programs(lines, k, iters=25, tol=1e-4):
    """Uncentred PCA: the top-k right singular vectors of the stacked knockdown x gene table (own gene = 0).
    Genes a line did not measure are missing, not zero: they are filled from the current rank-k fit and the SVD is
    redone (EM-PCA) until the fill stops changing. With nothing missing this is one plain SVD."""
    from sklearn.utils.extmath import randomized_svd
    Y = np.vstack([own_set(l.lfc, l.own, 0.0) for l in lines])
    observed = np.isfinite(Y); filled = np.where(observed, Y, 0).astype(np.float32)
    for _ in range(iters if not observed.all() else 1):
        u, s, vt = randomized_svd(filled, n_components=k, n_iter=7, random_state=SEED)
        if observed.all(): break
        fit = ((u * s) @ vt).astype(np.float32)
        change = float(np.sqrt(np.mean((fit[~observed] - filled[~observed]) ** 2)))
        filled[~observed] = fit[~observed]
        if change < tol: break
    total = float(np.square(filled, dtype=np.float64).sum())
    return vt.T.astype(np.float32), s, total, filled


def scores(values, W, own=None):
    """Least-squares program scores of each row on the genes it has (own gene counted as 0). NaN rows -> NaN."""
    Y = own_set(values, own, 0.0) if own is not None else np.asarray(values, np.float32)
    obs = np.isfinite(Y); out = np.full((len(Y), W.shape[1]), np.nan, np.float32)
    if not len(Y): return out
    _, group = np.unique(np.packbits(obs, axis=1), axis=0, return_inverse=True)
    for g in np.unique(group):
        r = np.flatnonzero(group.ravel() == g); m = obs[r[0]]
        if m.sum() < W.shape[1]: continue                   # too few genes to place the row
        Wm = W[m].astype(np.float64)
        out[r] = np.linalg.solve(Wm.T @ Wm, Wm.T @ Y[np.ix_(r, m)].T.astype(np.float64)).T
    return out


# ---------------------------------------------------------------- Model 2 baseline without the destination
def model2_tau2(sources_dir, all_genes, excluded):
    """Model 2's per-gene tau2, refit with the excluded backgrounds removed (same rule as vcc2_meta.fit)."""
    sources = [(n, b) for n, b in meta.included(sources_dir, LINES) if b not in excluded]
    panel = meta.Panel(sources_dir, sources, all_genes)
    tau2, _ = meta.tau2_dl(panel.lfc, panel.se, panel.own_col)
    est = np.isfinite(tau2)
    return np.where(est, tau2, np.median(tau2[est]))


def model2_rows(targets, own, sources, tau2, chunk=1024):
    """Model 2 prediction for `targets` (own gene column per target, -1 = none) from `sources` only."""
    T, G = len(targets), len(tau2)
    out = np.full((T, G), np.nan, np.float32)
    rows = [np.array([o.row.get(t, -1) for t in targets]) for o in sources]
    for start in range(0, T, chunk):
        idx = np.arange(start, min(start + chunk, T))
        lfc = np.full((len(sources), len(idx), G), np.nan, np.float32); se = lfc.copy()
        for s, (o, r) in enumerate(zip(sources, rows)):
            have = r[idx] >= 0
            lfc[s, have] = o.lfc[r[idx][have]]; se[s, have] = o.se[r[idx][have]]
        o_ = own[idx]; j = np.flatnonzero(o_ >= 0)
        lfc[:, j, o_[j]] = np.nan
        out[idx] = meta.combine(lfc, se, tau2)[0]
    return out


def model2_baseline(dest, others, tau2, chunk=1024):
    """Model 2 prediction of every knockdown of `dest` from the other lines only (own gene NaN)."""
    return model2_rows(dest.targets, dest.own, others, tau2, chunk)


def signal_k(lines, share=SIGNAL_SHARE, seed=SEED, filled=None):
    """k80: the smallest k whose components hold `share` of the variance above the noise (parallel analysis).

    The noise spectrum comes from a table of pure measurement noise of the same shape, each entry drawn N(0, SE^2)
    with its own SE (own gene 0 in both). A component carries signal when its eigenvalue exceeds the LARGEST noise
    eigenvalue (the noise edge; comparing rank by rank is biased, since signal components push the data's noise bulk
    down the ranks). k80 accumulates the signal components' excess over the noise edge until `share` is reached."""
    Y = np.vstack([own_set(l.lfc, l.own, 0.0) for l in lines]).astype(np.float64)
    if filled is not None: Y = filled.astype(np.float64)                      # EM-filled entries carry no noise
    sd = np.vstack([own_set(l.se, l.own, 0.0) for l in lines]).astype(np.float64)
    sd[~np.isfinite(sd)] = 0
    data = np.clip(np.linalg.eigvalsh(Y.T @ Y)[::-1], 0, None)
    noise_table = np.random.default_rng(seed_for(seed, 'signal_k')).standard_normal(Y.shape) * sd; del Y, sd
    noise = np.clip(np.linalg.eigvalsh(noise_table.T @ noise_table)[::-1], 0, None); del noise_table
    edge = float(noise[0])
    n_signal = int((data > edge).sum())
    excess = data[:n_signal] - edge
    cumulative = np.cumsum(excess) / excess.sum() if n_signal else np.array([1.0])
    k80 = int(np.searchsorted(cumulative, share) + 1) if n_signal else 1
    return dict(k80=k80, share=share, n_signal_components=n_signal, noise_edge=edge, noise_share_of_total=float(noise.sum() / data.sum()),
                signal_share_kept={int(k): float(cumulative[min(k, len(cumulative)) - 1]) for k in (*K_GRID, k80)})


# ---------------------------------------------------------------- ridge
def ridge_fit(X, y, penalty):
    """Ridge on standardised inputs; the intercept is not penalised. Loss: mean squared error + penalty x |b|^2."""
    mu = X.mean(0); sd = X.std(0); sd[sd == 0] = 1
    Z = (X - mu) / sd; n = len(y)
    coef = np.linalg.solve(Z.T @ Z / n + penalty * np.eye(X.shape[1]), Z.T @ (y - y.mean()) / n)
    return dict(mu=mu, sd=sd, coef=coef, intercept=float(y.mean()), penalty=float(penalty))


def ridge_predict(fit, X):
    return ((X - fit['mu']) / fit['sd']) @ fit['coef'] + fit['intercept']


def folds_for(targets, folds=FOLDS):
    """Fold of each pair by its knockdown: every pair of one knockdown lands in the same fold."""
    return np.array([seed_for(SEED, 'fold', t) % folds for t in targets])


def fit_program(X, y, fold, penalties=PENALTIES):
    """Pick the penalty by grouped CV, refit on all pairs. Returns (fit, cv table, out-of-fold predictions)."""
    cv, oof = {}, {}
    for lam in penalties:
        pred = np.empty(len(y))
        for f in np.unique(fold):
            test = fold == f
            pred[test] = ridge_predict(ridge_fit(X[~test], y[~test], lam), X[test])
        cv[lam] = float(np.mean((pred - y) ** 2)); oof[lam] = pred
    best = min(cv, key=cv.get)
    return ridge_fit(X, y, best), cv, oof[best]


def features(slot_scores, slot_missing, j, extra=None):
    """Inputs of program j: each slot's score (0 when missing) and missing flag, plus an optional extra column."""
    cols = [np.where(slot_missing, 0.0, np.nan_to_num(slot_scores[:, :, j])), slot_missing.astype(float)]
    if extra is not None: cols.append(extra[:, [j]])
    return np.hstack(cols)


# ---------------------------------------------------------------- fit
def slot_inputs(targets, lines, W, cache, exclude=None):
    """Scores of every slot line for `targets`: (n x slots x k) and missing flags (n x slots)."""
    n, k = len(targets), W.shape[1]
    S = np.zeros((n, len(lines), k), np.float32); missing = np.ones((n, len(lines)), bool)
    for s, line in enumerate(lines):
        if line.name == exclude: continue
        r = np.array([line.row.get(t, -1) for t in targets]); have = r >= 0
        sc = cache[line.name][r[have]]
        ok = np.isfinite(sc).all(1); idx = np.flatnonzero(have)[ok]
        S[idx, s] = sc[ok]; missing[idx, s] = False
    return S, missing


def fit(filters_dir, sources_dir, genes_csv, output, holdout=None, k_grid=K_GRID, signal_share=SIGNAL_SHARE):
    check_backgrounds(sources_dir)
    all_genes = read_names(genes_csv, 'gene_name')
    names = [n for n in LINES if BACKGROUND[n] != holdout]
    pgenes = program_genes(filters_dir, all_genes)
    print(f'fit: lines {names} (held out: {holdout}); {len(pgenes):,} program genes', flush=True)
    lines = [Line(filters_dir, n, pgenes) for n in names]
    program_lines = [l for l in lines if l.name in PROGRAM_SOURCES]
    print(f"  program sources: {[l.name for l in program_lines]}", flush=True)
    W_all, sing, total, filled = fit_programs(program_lines, max(k_grid))
    signal = None
    if signal_share:
        signal = signal_k(program_lines, signal_share, filled=filled)
        print(f"  k80: {signal['k80']} programs hold {signal_share:.0%} of the above-noise variance "
              f"({signal['n_signal_components']} components above noise; noise = {signal['noise_share_of_total']:.0%} of the total)", flush=True)
        if signal['k80'] < max(k_grid) and signal['k80'] not in k_grid:   # never above the grid's ceiling
            k_grid = tuple(sorted(set(k_grid) | {signal['k80']}))
    del filled
    explained = (sing ** 2 / total).tolist()
    print('  variance explained by the top programs: ' + ', '.join(f'k={k}: {sum(explained[:k]):.1%}' for k in k_grid), flush=True)

    # Destination rows and the Model 2 baseline built without each destination.
    gpos = {g: i for i, g in enumerate(all_genes)}; pidx = np.array([gpos[g] for g in pgenes])
    pairs = []
    for d in lines:
        others = [o for o in lines if o is not d]
        kept_elsewhere = np.array([any(t in o.row for o in others) for t in d.targets])
        if not kept_elsewhere.any(): continue
        tau2 = model2_tau2(sources_dir, all_genes, {holdout, d.background} - {None})[pidx]
        base = model2_baseline(d, others, tau2)
        rows = np.flatnonzero(kept_elsewhere)
        pairs.append(dict(dest=d, rows=rows, base=base[rows]))
        print(f'  destination {d.name}: {len(rows):,} knockdowns kept in >= 1 other line', flush=True)
    for l in lines: l.se = None                                  # only the baselines needed it

    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    np.save(out / 'program_genes.npy', pgenes); np.save(out / 'loadings.npy', W_all)
    report, params = [], {}
    for k in k_grid:
        W = W_all[:, :k]
        cache = {l.name: scores(l.lfc, W, l.own) for l in lines}
        X_rows = {'programs': [], 'corrected': []}; Y_rows = {'programs': [], 'corrected': []}; meta_rows = []
        for p in pairs:
            d, rows = p['dest'], p['rows']; targets = d.targets[rows]
            S, missing = slot_inputs(targets, lines, W, cache, exclude=d.name)
            target_scores = cache[d.name][rows]
            resid = own_set(d.lfc[rows] - p['base'], d.own[rows], 0.0)
            base_scores = scores(p['base'], W, d.own[rows])
            resid_scores = scores(resid, W)
            ok = np.isfinite(target_scores).all(1) & np.isfinite(resid_scores).all(1) & np.isfinite(base_scores).all(1)
            meta_rows.append(dict(dest=d, rows=rows[ok], targets=targets[ok], base=p['base'][ok]))
            X_rows['programs'].append((S[ok], missing[ok], None)); Y_rows['programs'].append(target_scores[ok])
            X_rows['corrected'].append((S[ok], missing[ok], base_scores[ok])); Y_rows['corrected'].append(resid_scores[ok])
        targets_all = np.concatenate([m['targets'] for m in meta_rows]); fold = folds_for(targets_all)
        params[k] = {}
        for variant in ['programs', 'corrected']:
            S = np.concatenate([x[0] for x in X_rows[variant]]); M = np.concatenate([x[1] for x in X_rows[variant]])
            extra = None if variant == 'programs' else np.concatenate([x[2] for x in X_rows[variant]])
            Y = np.concatenate(Y_rows[variant]); oof = np.empty_like(Y); fits = []
            for j in range(k):
                f, cv, oof[:, j] = fit_program(features(S, M, j, extra), Y[:, j], fold)
                fits.append(f)
            params[k][variant] = fits
            report += cv_report(k, variant, oof, Y, W, meta_rows)
        print(f'  k={k}: {len(targets_all):,} training pairs, {len(np.unique(targets_all)):,} knockdowns', flush=True)
    save_params(out / 'ridge.npz', params)
    frame = pd.DataFrame(report); frame.to_csv(out / 'cv_report.csv', index=False)
    model = dict(schema=SCHEMA, filters_dir=str(Path(filters_dir).resolve()), sources_dir=str(Path(sources_dir).resolve()),
                 genes_csv=str(Path(genes_csv).resolve()), heldout_background=holdout, lines=names,
                 backgrounds=[BACKGROUND[n] for n in names], program_sources=[l.name for l in program_lines],
                 gene_set_sources=GENE_SET_SOURCES, k_grid=list(k_grid),
                 penalties=list(PENALTIES), folds=FOLDS, n_program_genes=len(pgenes),
                 variance_explained={k: float(sum(explained[:k])) for k in k_grid},
                 own_gene='set to 0 in programs and scores; handled by ontarget_lfc',
                 cv='grouped by knockdown: a selection estimate for unseen knockdowns in known lines, not a transfer estimate',
                 signal_k=signal)
    save_json(out / 'model.json', model)
    summary = frame[frame.destination == 'all'].pivot(index='k', columns='variant', values='gene_mse')
    print('  CV gene-space MSE on held-out knockdowns (all destinations):'); print(summary.round(5).to_string())
    return model


def cv_report(k, variant, oof, Y, W, meta_rows):
    """Gene-space error of out-of-fold predictions vs Model 2 and vs predicting no change."""
    out, start, totals = [], 0, dict(n=0, model2=0.0, zero=0.0, pred=0.0)
    for m in meta_rows:
        n = len(m['rows']); d = m['dest']; s = oof[start:start + n]; start += n
        measured = own_set(d.lfc[m['rows']], d.own[m['rows']], np.nan)
        rebuilt = s @ W.T
        pred = rebuilt if variant == 'programs' else m['base'] + rebuilt
        ok = np.isfinite(measured) & np.isfinite(pred) & np.isfinite(m['base'])
        e = dict(n=int(ok.sum()), model2=float(((measured - m['base'])[ok] ** 2).sum()),
                 zero=float((measured[ok] ** 2).sum()), pred=float(((measured - pred)[ok] ** 2).sum()))
        for key in totals: totals[key] += e[key]
        out.append(dict(k=k, variant=variant, destination=d.name, pairs=n, gene_mse=e['pred'] / e['n'],
                        model2_mse=e['model2'] / e['n'], no_change_mse=e['zero'] / e['n']))
    score_r2 = 1 - ((oof - Y) ** 2).sum(0) / np.maximum(((Y - Y.mean(0)) ** 2).sum(0), 1e-12)
    out.append(dict(k=k, variant=variant, destination='all', pairs=int(len(Y)), gene_mse=totals['pred'] / totals['n'],
                    model2_mse=totals['model2'] / totals['n'], no_change_mse=totals['zero'] / totals['n'],
                    score_r2_mean=float(score_r2.mean())))
    return out


def save_params(path, params):
    flat = {}
    for k, variants in params.items():
        for variant, fits in variants.items():
            for j, f in enumerate(fits):
                for key, value in f.items(): flat[f'k{k}/{variant}/{j}/{key}'] = np.asarray(value)
    np.savez(path, **flat)


def load_params(path, k, variant):
    z = np.load(path); fits = []
    for j in range(k):
        fits.append({key: z[f'k{k}/{variant}/{j}/{key}'] for key in ['mu', 'sd', 'coef', 'intercept', 'penalty']})
    return fits


# ---------------------------------------------------------------- predict
def predict_scores(fits, S, missing, extra):
    return np.stack([ridge_predict(f, features(S, missing, j, extra)) for j, f in enumerate(fits)], 1)


def leave_one_slot_out(fits, S, missing, extra, extra_by_slot=None):
    """Average masked predictions only for full rows; spread is in program space.

    Spread has one row per target, with NaN for rows not subject to averaging. For corrected, `extra_by_slot[s]`
    holds the full rows' Model 2 scores rebuilt WITHOUT slot s (as in training); without it, `extra` is reused.
    """
    before = predict_scores(fits, S, missing, extra)
    after = before.copy()
    full = ~missing.any(1)
    spread = np.full(before.shape, np.nan, dtype=before.dtype)
    if full.any():
        loo = []
        for slot in range(S.shape[1]):
            m = missing[full].copy(); m[:, slot] = True
            X = S[full].copy(); X[:, slot] = 0
            e = None if extra is None else (extra_by_slot[slot] if extra_by_slot is not None else extra[full])
            loo.append(predict_scores(fits, X, m, e))
        loo = np.stack(loo)
        after[full] = loo.mean(0)
        spread[full] = loo.std(0)
    return before, after, spread, full


def mean_abs_delta(delta):
    finite = np.isfinite(delta)
    return float(np.abs(delta[finite]).mean(dtype=np.float64)) if finite.any() else None


def predict(model_dir, baseline_dir, lines_out, output):
    m = Path(model_dir); model = json.loads((m / 'model.json').read_text())
    check_backgrounds(model['sources_dir'])
    if model.get('schema') != SCHEMA: raise ValueError('incompatible model; fit with this notebook')
    b = Path(baseline_dir); report2 = json.loads((b / 'prediction_report.json').read_text())
    if report2.get('heldout_background') != model['heldout_background']:
        raise ValueError(f'baseline held out {report2.get("heldout_background")}, model held out {model["heldout_background"]}')
    pgenes = np.load(m / 'program_genes.npy').astype(str); W_all = np.load(m / 'loadings.npy')
    genes = np.load(b / 'genes.npy').astype(str); targets = np.load(b / 'targets.npy').astype(str)
    re = np.load(b / 're.npy'); gpos = {g: i for i, g in enumerate(genes)}; pidx = np.array([gpos[g] for g in pgenes])
    lines = [Line(model['filters_dir'], n, pgenes) for n in model['lines']]
    own_p = np.array([{g: i for i, g in enumerate(pgenes)}.get(t, -1) for t in targets])
    own_all = np.array([gpos.get(t, -1) for t in targets])
    out = Path(output)
    for line in lines_out:
        d = out / line; earlier = d / 'prediction_report.json'
        if d.exists() and any(d.iterdir()) and not (earlier.exists() and json.loads(earlier.read_text()).get('schema') == SCHEMA):
            raise ValueError(f'choose an empty prediction directory: {d}')
    arrays, coverage, spreads, averaging = {}, {}, {}, {}
    # Rows with every slot present: Model 2 rebuilt without each slot in turn (tau2 refit), for the corrected masks.
    present = np.array([[t in l.row for l in lines] for t in targets])
    full_rows = np.flatnonzero(present.all(1))
    all_genes = read_names(model['genes_csv'], 'gene_name')
    without_slot = []
    if len(full_rows):
        for s_, line in enumerate(lines):
            tau2 = model2_tau2(model['sources_dir'], all_genes, {model['heldout_background'], line.background} - {None})
            tau2 = tau2[np.array([{g: i for i, g in enumerate(all_genes)}[g] for g in pgenes])]
            without_slot.append(model2_rows(targets[full_rows], own_p[full_rows], [l for l in lines if l is not line], tau2))
    for k in model['k_grid']:
        W = W_all[:, :k]
        cache = {l.name: scores(l.lfc, W, l.own) for l in lines}
        S, missing = slot_inputs(targets, lines, W, cache)
        none = missing.all(1)                                    # no line kept this knockdown: no information
        base_scores = scores(re[:, pidx], W, own_p)
        for variant in ['programs', 'corrected']:
            fits = load_params(m / 'ridge.npz', k, variant)
            extra = None if variant == 'programs' else np.nan_to_num(base_scores)
            extra_by_slot = None
            full = ~missing.any(1)
            sel = np.searchsorted(full_rows, np.flatnonzero(full))      # full rows are a subset of the rows with every line
            if variant == 'corrected' and full.any():
                assert np.isin(np.flatnonzero(full), full_rows).all(), 'every full row needs its per-slot baselines'
                extra_by_slot = [np.nan_to_num(scores(b[sel], W, own_p[full])) for b in without_slot]
            before, s, spread, full = leave_one_slot_out(fits, S, missing, extra, extra_by_slot)
            rebuilt = (s @ W.T).astype(np.float32)
            if variant == 'programs':
                delta = np.full(re.shape, np.nan, np.float32); delta[:, pidx] = rebuilt
            else:
                delta = re.copy(); delta[:, pidx] = re[:, pidx] + rebuilt   # NaN baseline stays NaN
                if full.any():
                    # each mask predicts the residual of Model 2 WITHOUT that slot: average (that baseline + correction)
                    with warnings.catch_warnings():              # genes no remaining line measured stay NaN
                        warnings.simplefilter('ignore', RuntimeWarning)
                        base_avg = np.nanmean(np.stack([b[sel] for b in without_slot]), 0)
                    delta[np.ix_(np.flatnonzero(full), pidx)] = base_avg + rebuilt[full]
            delta[none] = np.nan
            r = np.flatnonzero(own_all >= 0); delta[r, own_all[r]] = np.nan
            name = f'{variant}_k{k}'
            arrays[name] = delta
            spreads[f'{name}_spread'] = spread
            prior = delta[full].copy()
            raw_rebuilt = (before[full] @ W.T).astype(np.float32)
            prior[:, pidx] = raw_rebuilt if variant == 'programs' else re[full][:, pidx] + raw_rebuilt
            prior = own_set(prior, own_all[full], np.nan)
            before_abs, after_abs = mean_abs_delta(prior), mean_abs_delta(delta[full])
            # The correction alone (programs x predicted scores), own gene excluded: for corrected, |delta| is mostly Model 2.
            corr_before = mean_abs_delta(own_set(raw_rebuilt, own_p[full], np.nan)) if full.any() else None
            corr_after = mean_abs_delta(own_set(rebuilt[full], own_p[full], np.nan)) if full.any() else None
            averaging[name] = dict(affected_rows=int(full.sum()), affected_targets=targets[full].tolist(),
                                   mean_abs_delta_before=before_abs, mean_abs_delta_after=after_abs,
                                   mean_abs_correction_before=corr_before, mean_abs_correction_after=corr_after,
                                   after_smaller=None if corr_before is None else corr_after < corr_before,
                                   corrected_baseline='Model 2 rebuilt without each masked slot' if variant == 'corrected' else None)
            print(f'  {name}: leave-one-slot-out on {full.sum()} rows; mean |correction| {corr_before} -> {corr_after}', flush=True)
        coverage[k] = int((~none).sum())
    models = ['re'] + list(arrays) + ['zero']
    report = dict(schema=SCHEMA, models=models, heldout_background=model['heldout_background'],
                  source_backgrounds=model['backgrounds'], model_dir=str(m.resolve()), baseline_dir=str(b.resolve()),
                  re='Model 2 (vcc2_meta.py), copied unchanged; re_se is its SE',
                  zero='all-zero effects with the same shape as re; the generator still applies the knockdown',
                  programs='loadings x ridge-predicted program scores on the program genes; NaN elsewhere',
                  corrected='Model 2 + loadings x ridge-predicted residual scores on the program genes; Model 2 elsewhere',
                  se='none for the ridge outputs: use the generator with --z-grid 0',
                  leave_one_slot_out=averaging,
                  averaging_metric='mean absolute effect over finite output genes of affected rows; own gene excluded',
                  spread='population SD across masked predictions in program space; targets x k, NaN on unaffected rows; not an SE',
                  n_program_genes=len(pgenes), targets_with_any_line=coverage[model['k_grid'][0]],
                  no_information=[t for t, n in zip(targets.tolist(), missing.all(1)) if n])
    for line in lines_out:
        d = out / line; d.mkdir(parents=True, exist_ok=True)
        for old in [*d.glob('*.npy'), d / 'prediction_report.json']: old.unlink(missing_ok=True)
        for name in ['genes', 'targets', 're', 're_se', 'n_backgrounds', 'ontarget_lfc']:
            np.save(d / f'{name}.npy', np.load(b / f'{name}.npy'))
        for name in ['re_z', 'n_bg', 'sign_agree']:              # Model 2's ranking and sign-agreement inputs, when present
            if (b / f'{name}.npy').exists(): np.save(d / f'{name}.npy', np.load(b / f'{name}.npy'))
        for name, value in arrays.items(): np.save(d / f'{name}.npy', value)
        for name, value in spreads.items(): np.save(d / f'{name}.npy', value)
        np.save(d / 'zero.npy', np.zeros_like(re))
        save_json(d / 'prediction_report.json', dict(report, line=line))
    print(f'predict: {len(targets)} targets -> {", ".join(str(out / l) for l in lines_out)}; models {models}')
    print(f'  {report["targets_with_any_line"]} targets kept in >= 1 line; no information: {report["no_information"] or "none"}')
    return report


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)
    f = sub.add_parser('fit')
    for key in ['filters-dir', 'sources-dir', 'genes', 'out']: f.add_argument('--' + key, required=True)
    f.add_argument('--holdout'); f.add_argument('--k-grid', nargs='+', type=int, default=list(K_GRID))
    f.add_argument('--signal-share', type=float, default=SIGNAL_SHARE, help='k80: share of the above-noise variance; 0 = grid only')
    pr = sub.add_parser('predict')
    for key in ['model', 'baseline', 'out']: pr.add_argument('--' + key, required=True)
    pr.add_argument('--lines', nargs='+', required=True)
    a = p.parse_args()
    if a.cmd == 'fit': fit(a.filters_dir, a.sources_dir, a.genes, a.out, a.holdout, tuple(a.k_grid), a.signal_share)
    else: predict(a.model, a.baseline, a.lines, a.out)

### vcc3_generate.py
`vcc_2026_2a`'s generator and leaderboard-scored benchmark, plus `gate_from` (call genes with Model 2's significance, keep another array's values) and the `nochange` pseudo-model (zero effects, no knockdown).

In [ ]:
%%writefile ~/vcc/code/vcc_2026_3c/vcc3_generate.py
"""Generate counts with projected composition, metadata cell counts/library sizes,
state-dependent overdispersed induction and reproducible common control sampling.

Finishing: a DE call, then one global alpha, an optional detection floor, the > 5 CPM gate in the line's own controls,
and the knocked-down gene applied separately. Two calling rules:
  z     keep |delta| > z* x SE_pred and |delta| > a minimum size (the vcc_2026_2 submission). With gate_from (e.g.
        're'), the keep-mask comes from that array and its SE (|re| > z* x re_se) and is applied to the selected
        array's values, so a ridge output without its own SE can borrow Model 2's significance call
  topn  per knockdown keep the N genes with the largest |delta| / SE_pred among genes > 5 CPM (optionally only genes
        whose backgrounds agree in sign, or measured in >= min_bg backgrounds); every other gene is 0. With gate_from, that array's |delta| / SE ranks the
        genes and the selected array supplies the values (e.g. Model 2 picks, the corrected ridge output is used)
Detection floor: each called gene is raised to the smallest log2 change a 400-cell test against the line's controls
would detect at z_req, so that the genes we call actually show up as DE in the generated cells.

Pseudo-model `nochange`: zero effects AND no knockdown (the target gene is left at its control level), the true
no-change arm; the `zero` array instead keeps the measured knockdown.

`benchmark` scores arms on the exam (H1-held-out validation) with official cell-eval2 (vcc2026), streaming every
file one target at a time. The real side, the ceiling and the competition bundle (generic baseline + replicate anchor)
are built once, and can be reused by a later stage (--reference). Every arm is scored with the leaderboard formula
(scaled, clamped, averaged: scaled_scores.csv) and selection uses that overall; raw metrics and ranks are kept.
`oracle` is a separate generator ceiling check and never selects submission knobs. With --variants it scores several
arms on the same real cells (method `oracle` or `nochange`, any alpha and cell rule); with --scale it also builds the
competition bundle on those cells, so the ceiling is reported on the leaderboard scale.
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import shlex
import subprocess
import shutil
import copy
from vcc2_common import int64_indptr, seed_for, validate_counts, save_json, expression_gate, moments, alignment

try:
    from anndata.io import read_elem, write_elem, sparse_dataset
except ImportError:  # older anndata
    from anndata.experimental import read_elem, write_elem, sparse_dataset

VCC2026_METRICS = {
    'pds_cosine': 'max', 'expr_mse_unbiased_capped_norm': 'min',
    'de_wilcoxon_lfc_nmae': 'min', 'de_wilcoxon_direction_fidelity_yield_raw': 'max',
    'de_wilcoxon_direction_reach_raw': 'max', 'de_wilcoxon_sig_jaccard': 'max',
}
CONTROL = "non-targeting"
SCHEMA = "vcc_2026_3_ridge_v1"
CC_GENES = ("MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8 HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 "
            "CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 "
            "NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA").split()


class Cfg:
    def __init__(self, a):
        self.alpha = getattr(a, "alpha", 1.0); self.max_lfc = 8.0; self.min_lfc = 0.0
        self.kd_clip = (0.01, 0.9); self.default_kd = 0.30; self.n_cells = getattr(a, "n_cells", 400)
        self.composition = "projection"; self.comp_thr = getattr(a, "comp_thr", 0.02)
        self.pi = getattr(a, "pi", 0.0); self.up_w = getattr(a, "up_w", 0.75)
        self.tilt = getattr(a, "tilt", False); self.match_pool = getattr(a, "match_pool", False) or self.tilt
        self.seed = getattr(a, "seed", 2026)
        self.z = getattr(a, "z", 0.0); self.min_effect = getattr(a, "min_effect", 0.0)   # z call; 0 and 0 = off
        self.call = getattr(a, "call", "z"); self.top_n = int(getattr(a, "top_n", 1000))
        self.sign_rule = bool(getattr(a, "sign_rule", False)); self.detect_floor = bool(getattr(a, "detect_floor", False))
        self.z_req = float(getattr(a, "z_req", 4.0))
        self.min_bg = int(getattr(a, "min_bg", 1))                # top-N: only genes measured in >= min_bg backgrounds
        self.gate_from = getattr(a, "gate_from", "") or ""
        self.metadata = getattr(a, "metadata", True)
        self.library_size = None
        if (not 0 <= self.pi < 1 or not 0 <= self.up_w <= 1 or self.n_cells < 1 or self.alpha < 0
                or self.z < 0 or self.min_effect < 0 or self.call not in ("z", "topn") or self.top_n < 1 or self.z_req <= 0 or self.min_bg < 1
                or (self.gate_from and self.call == "z" and self.z <= 0)):
            raise ValueError("invalid generator settings")

    def as_dict(self):
        # min_bg is left out at its default, so settings written before it existed still describe the same arm.
        return {k: v for k, v in vars(self).items() if not (k == "min_bg" and v == 1)}


# ---------------------------------------------------------------- inputs
def read_list(path, col):
    return pd.read_csv(path)[col].astype(str).tolist()


def aligned_X(a, genes):
    """cells x our genes, CSR float32 (missing genes = 0, extra genes dropped)."""
    X = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)
    names = a.var_names.astype(str).tolist()
    if names == list(genes):
        return X.astype(np.float32)
    pos = {g: i for i, g in enumerate(genes)}
    src = np.array([i for i, g in enumerate(names) if g in pos]); dst = np.array([pos[names[i]] for i in src])
    X = X[:, src].tocoo()
    print(f"  re-indexed genes: {len(src):,}/{len(genes):,} of ours present")
    return sp.csr_matrix((X.data.astype(np.float32), (X.row, dst[X.col])), shape=(X.shape[0], len(genes)))


class Line:
    """Control cells of one context plus the statistics the generator needs."""
    def __init__(self, X, guide, genes):
        validate_counts(X, "control cells")
        self.X = X.tocsr().astype(np.float32); self.genes = genes; self.n = X.shape[0]
        if not self.n or self.X.sum() <= 0: raise ValueError("controls must contain cells and positive counts")
        self.lib = np.asarray(self.X.sum(1)).ravel()
        self.cpm_cells = sp.diags(1e6 / np.maximum(self.lib.astype(np.float64), 1)) @ self.X
        self.cpm = np.asarray(self.cpm_cells.mean(0)).ravel()           # mean per-cell CPM (scorer gate uses > 5)
        self.cpm_var = np.maximum(np.asarray(self.cpm_cells.multiply(self.cpm_cells).mean(0)).ravel() - self.cpm ** 2, 0)
        self.n_reference = self.n                                       # control cells the scorer compares against
        self.share = np.asarray(self.X.sum(0)).ravel() / self.X.sum()   # pooled share of molecules
        self.gate = expression_gate(self.cpm)
        count_mean = np.asarray(self.X.mean(0)).ravel()
        count_var = np.maximum(np.asarray(self.X.multiply(self.X).mean(0)).ravel()-count_mean**2, 0)
        self.overdispersion = np.where(count_mean > 0, np.clip((count_var-count_mean)/np.maximum(count_mean**2, 1e-12), 0, 10), 1.0)
        g = np.asarray(guide).astype(str) if guide is not None else np.zeros(self.n, str)
        self.groups = [np.flatnonzero(g == u) for u in np.unique(g)]
        gp = {x: i for i, x in enumerate(genes)}
        self.cc_idx = np.array([gp[x] for x in CC_GENES if x in gp and self.cpm[gp[x]] > 1], int)
        if len(self.cc_idx) >= 5:
            L = self.cpm_cells[:, self.cc_idx].toarray()
            self.cyc = np.log2(L + 1).mean(1)                            # per-cell cycling level (log2 units)
        else:
            self.cyc = None

    @staticmethod
    def from_h5ad(path, genes, control_col="target_gene"):
        a = ad.read_h5ad(path)
        validate_counts(a.X, str(path))
        if control_col and control_col in a.obs:
            a = a[a.obs[control_col].astype(str).values == CONTROL].copy()
        guide = next((a.obs[c].values for c in ("ntc_id", "guide_id", "sgID_AB", "guide") if c in a.obs), None)
        return Line(aligned_X(a, genes), guide, genes)


def load_deltas(path, targets, genes, array="re", fill=0.0):
    """meta prediction dir (genes.npy, targets.npy, <array>.npy) or .npz (delta [+targets, genes]) -> T x G.

    Missing targets/genes and non-finite values become `fill`: 0 (no change) for deltas, inf (unknown) for SEs.
    """
    p = Path(path)
    if p.is_dir():
        D = np.load(p / f"{array}.npy", mmap_mode="r"); t = np.load(p / "targets.npy").astype(str)
        g = np.load(p / "genes.npy").astype(str)
    else:
        z = np.load(p, allow_pickle=False); D = z["delta"]
        t = z["targets"].astype(str) if "targets" in z else np.array(targets)
        g = z["genes"].astype(str) if "genes" in z else np.array(genes)
    out = np.full((len(targets), len(genes)), fill, np.float32)
    tp = {x: i for i, x in enumerate(t)}; gp = {x: i for i, x in enumerate(g)}
    gsrc = np.array([gp.get(x, -1) for x in genes]); ok = gsrc >= 0
    missing = 0
    for i, x in enumerate(targets):
        if x in tp:
            out[i, ok] = np.asarray(D[tp[x]])[gsrc[ok]]
        else:
            missing += 1
    if missing:
        print(f"  WARNING {path}: {missing} targets missing -> no change predicted for them")
    out[~np.isfinite(out)] = fill
    return out


def load_info(path, targets, genes, model, gate_from=""):
    """The calling inputs that exist next to a model's table: se, z (|delta| / SE), n_bg and sign_agree, plus the
    gating array and its SE when another array decides which genes are called (gate_from)."""
    p = Path(path); info = {}
    for key, name, fill in [("se", model + "_se", np.inf), ("z", model + "_z", np.nan), ("n_bg", "n_bg", np.nan),
                            ("sign_agree", "sign_agree", np.nan)]:
        if p.is_dir() and (p / f"{name}.npy").exists(): info[key] = load_deltas(path, targets, genes, name, fill=fill)
    if gate_from:
        for key, name, fill in [("gate_delta", gate_from, 0.0), ("gate_se", gate_from + "_se", np.inf)]:
            if not (p / f"{name}.npy").exists(): raise ValueError(f"gate_from={gate_from!r} needs {name}.npy")
            info[key] = load_deltas(path, targets, genes, name, fill=fill)
    return info


def is_nochange(model):
    return model == "nochange"


def load_effects(path, targets, genes, model, cfg):
    """Delta table of one model, plus the calling inputs (SE, z, n_bg, sign_agree, gate) that exist for it.
    `nochange` is all zeros (and the caller skips the knockdown)."""
    if is_nochange(model): return np.zeros((len(targets), len(genes)), np.float32), {}
    D = load_deltas(path, targets, genes, model)
    info = load_info(path, targets, genes, model, cfg.gate_from)
    if cfg.z > 0 and not cfg.gate_from and "se" not in info: raise ValueError(f"{model}: the z call needs {model}_se.npy")
    return D, info


def kd_from_sources(targets, genes, orion_paths=(), tcell_path=None, tcell_condition="Stim48hr", pseudo=0.5):
    """Knocked-down gene's own log2 drop per target, averaged across sources (M1 rule). NaN if no source has it."""
    gp = {g: i for i, g in enumerate(genes)}; vals = [[] for _ in targets]
    for path in orion_paths:
        with h5py.File(path, "r") as f:
            obs = read_elem(f["obs"]); names = list(obs.index.astype(str)); n = obs["n_cells"].values.astype(float)
            var = read_elem(f["var"]); vg = list(var.index.astype(str))
            vpos = {g: i for i, g in enumerate(vg)}
            c = names.index(CONTROL)
            want = [(i, t) for i, t in enumerate(targets) if t in names and t in vpos]
            if not want:
                continue
            rows = sorted({c} | {names.index(t) for _, t in want})
            S = f["layers/sum_cpm"][rows, :]; rpos = {r: k for k, r in enumerate(rows)}
            ctrl = S[rpos[c]] / n[c]
            for i, t in want:
                r = names.index(t); col = vpos[t]
                vals[i].append(np.log2((S[rpos[r], col] / n[r] + pseudo) / (ctrl[col] + pseudo)))
    if tcell_path:
        t = ad.read_h5ad(tcell_path)
        if "condition" in t.obs:
            t = t[t.obs["condition"].astype(str) == tcell_condition]
        tg = t.obs["target_gene"].astype(str).values; tv = {g: i for i, g in enumerate(t.var_names.astype(str))}
        X = np.asarray(t.X)
        for i, x in enumerate(targets):
            r = np.flatnonzero(tg == x)
            if len(r) and x in tv and np.isfinite(X[r[0], tv[x]]):
                vals[i].append(float(X[r[0], tv[x]]))
    return np.array([np.mean(v) if v else np.nan for v in vals])


def kd_residual(log2_drop, cfg):
    r = np.where(np.isfinite(log2_drop), 2.0 ** np.nan_to_num(log2_drop), cfg.default_kd)
    return np.clip(r, *cfg.kd_clip)


# ---------------------------------------------------------------- core generator
def call_top_n(D, line, own, cfg, info):
    """Per knockdown, keep the top_n genes by |delta| / SE among genes > 5 CPM (never the knocked-down gene).
    With sign_rule, a gene measured in >= 2 backgrounds is a candidate only if they all agree in sign.
    With gate_from, the gating array's |delta| / SE ranks the genes; the values stay those of D."""
    if cfg.gate_from:
        with np.errstate(divide="ignore", invalid="ignore"): z = np.abs(info["gate_delta"]) / info["gate_se"]
    else:
        z = info.get("z")
        if z is None:
            if "se" not in info: raise ValueError("the top-N call needs re_z or the prediction SE")
            with np.errstate(divide="ignore", invalid="ignore"): z = np.abs(D) / info["se"]
    cand = line.gate[None, :] & np.isfinite(D) & (D != 0) & np.isfinite(z)
    rows = np.flatnonzero(own >= 0); cand[rows, own[rows]] = False
    if cfg.sign_rule:
        if "n_bg" not in info or "sign_agree" not in info: raise ValueError("the sign rule needs n_bg and sign_agree")
        with np.errstate(invalid="ignore"):
            cand &= ~((info["n_bg"] >= 2) & (info["sign_agree"] < 1))
    if cfg.min_bg > 1:
        if "n_bg" not in info: raise ValueError("min_bg needs n_bg")
        with np.errstate(invalid="ignore"):
            cand &= info["n_bg"] >= cfg.min_bg
    score = np.where(cand, z, -np.inf)
    keep = np.zeros_like(cand)
    n = min(cfg.top_n, D.shape[1])
    top = np.argpartition(-score, n - 1, axis=1)[:, :n]
    np.put_along_axis(keep, top, True, axis=1)
    return np.where(keep & cand, D, 0).astype(np.float32)


def detection_floor(line, cfg, iters=40):
    """Per gene, the smallest |log2 change| (up, down) at which a cfg.n_cells-cell group would differ from the line's
    controls at z_req: m |f - 1| / sqrt(v f^2 / n_cells + v / n_reference) >= z_req, with f = 2^lfc and m, v the
    per-cell CPM mean and variance of the controls (the knockdown keeps the controls' CV). Unreachable -> max_lfc."""
    key = (cfg.n_cells, line.n_reference, cfg.z_req, cfg.max_lfc)
    cache = getattr(line, "_floor", {})
    if key in cache: return cache[key]
    m, v = line.cpm.astype(float), line.cpm_var.astype(float)
    def z_of(l):
        f = np.exp2(l)
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(v > 0, m * np.abs(f - 1) / np.sqrt(v * f ** 2 / cfg.n_cells + v / line.n_reference), np.inf)
    floors = []
    for end in (cfg.max_lfc, -cfg.max_lfc):                      # up, then down
        lo, hi = np.zeros_like(m), np.full_like(m, end)
        reachable = z_of(hi) >= cfg.z_req
        for _ in range(iters):                                    # bisection on the log2 scale between 0 and the cap
            mid = (lo + hi) / 2; ok = z_of(mid) >= cfg.z_req
            hi = np.where(ok, mid, hi); lo = np.where(ok, lo, mid)
        floors.append(np.where(reachable, np.abs(hi), cfg.max_lfc).astype(np.float32))
    cache[key] = tuple(floors); line._floor = cache
    return cache[key]


def calibrate(D, line, targets, cfg, se=None, info=None):
    """DE call, then the single global alpha, the optional detection floor and a binary responder-gene gate.

    z call: an effect stays at full size if |delta| > min_effect and, when z* > 0, |delta| > z* x SE_pred; else 0.
    topn call: see call_top_n. Unknown SE (inf/NaN) is never convincing. No shrinkage of retained effects.
    Detection floor: each called gene becomes sign(delta) x max(alpha |delta|, floor_g).
    The +/-8 generator limit is a numerical safety cap, not an estimated prior.
    On-target suppression is handled separately by measured guide efficiency.
    """
    info = dict(info or {})
    if se is not None: info.setdefault("se", se)
    D = np.asarray(D, np.float32)
    gp = {g: i for i, g in enumerate(line.genes)}
    own = np.array([gp.get(t, -1) for t in targets])
    if cfg.call == "topn":
        D = call_top_n(D, line, own, cfg, info)
    elif cfg.gate_from:
        # Model 2-gated: genes Model 2 calls (|re| > z* x re_se), with the selected array's values.
        with np.errstate(invalid="ignore"):
            keep = (np.abs(info["gate_delta"]) > cfg.z * np.asarray(info["gate_se"], np.float32)) & (np.abs(D) > cfg.min_effect)
        D = np.where(keep, D, 0)
    elif cfg.z > 0 or cfg.min_effect > 0:
        size = np.abs(D); keep = size > cfg.min_effect
        if cfg.z > 0:
            if "se" not in info: raise ValueError("the SE gate (z* > 0) needs the prediction SE")
            with np.errstate(invalid="ignore"):
                keep &= size > cfg.z * np.asarray(info["se"], np.float32)
        D = np.where(keep, D, 0)
    D = (cfg.alpha * D).astype(np.float32)
    if cfg.detect_floor:
        up, down = detection_floor(line, cfg)
        floor = np.where(D > 0, up[None, :], down[None, :])
        D = np.where(D != 0, np.sign(D) * np.maximum(np.abs(D), floor), 0)
    D = np.clip(D, -cfg.max_lfc, cfg.max_lfc).astype(np.float32)
    D[:, ~line.gate] = 0.0
    rows = np.flatnonzero(own >= 0); D[rows, own[rows]] = 0.0
    return D


def pick_cells(line, d, cfg, rng):
    """Step 3 (+ optional tilt): which control cells become this knockdown's 400 cells."""
    n = cfg.n_cells
    if cfg.tilt and line.cyc is not None:
        dcc = float(np.mean(d[line.cc_idx]))                       # predicted cycling shift (log2)
        v = line.cyc - line.cyc.mean(); s = max(v.std(), 1e-6)
        if abs(dcc) > 0.02:
            lo, hi = -30.0, 30.0
            for _ in range(50):                                     # bisection: weighted mean shift == dcc
                b = (lo + hi) / 2; w = np.exp(np.clip(b * v / s, -50, 50)); m = (w * v).sum() / w.sum()
                lo, hi = (b, hi) if m < dcc else (lo, b)
            w = np.exp(np.clip(b * v / s, -50, 50)); w /= w.sum()
            k = min(n, int((w > 0).sum()))
            return rng.choice(line.n, k, replace=False, p=w) if k == n else rng.choice(line.n, n, replace=True, p=w)
    if n > line.n:
        order = np.concatenate([rng.permutation(g) for g in line.groups])
        return np.concatenate([order, rng.choice(line.n, n-line.n, replace=True)])
    per = int(np.ceil(n / len(line.groups)))
    pick = np.concatenate([rng.choice(v, size=min(per, len(v)), replace=False) for v in line.groups])
    if len(pick) >= n:
        return rng.choice(pick, n, replace=False)
    rest = np.setdiff1d(np.arange(line.n), pick)
    return np.concatenate([pick, rng.choice(rest, n - len(pick), replace=len(rest) < n - len(pick))])


def project_factors(share, delta, j=None, kd=None):
    """Project all-gene factors onto the composition simplex without an LFC floor.

    The realized expectation is log2(projected_factor), not the unconstrained
    delta. Zero-baseline induction uses the same 0.5-CPM pseudocount as labels.
    """
    fac = np.exp2(np.asarray(delta, float))
    if j is not None and kd is not None: fac[j] = kd
    proposed = np.asarray(share, float)*fac
    zero_induced = (share == 0) & (fac > 1)
    proposed[zero_induced] = 0.5e-6*(fac[zero_induced]-1)
    mass = float(proposed.sum())
    if not np.isfinite(mass) or mass <= 0: raise ValueError("invalid projected composition")
    projected = proposed/mass
    factors = np.divide(projected, share, out=np.ones_like(projected), where=share > 0)
    return factors, projected, mass


def generate_one(line, d, j, kd, cfg, rng=None, pick=None, line_key="line", target_key="target"):
    """Integer counts, with stable per-gene streams shared between perturbations."""
    if rng is None: rng = np.random.default_rng(seed_for(cfg.seed, line_key, target_key))
    if pick is None:
        # Same base pool per line; target order does not advance a line-level stream.
        pick = pick_cells(line, d, cfg, np.random.default_rng(seed_for(cfg.seed, line_key, "base_controls")))
    B = line.X[pick].toarray().astype(np.int64); d = np.asarray(d, float).copy()
    if cfg.match_pool:
        sampled = np.asarray(line.cpm_cells[pick].mean(0)).ravel()
        deviation = np.log2((sampled+0.5)/(line.cpm+0.5))
        mask = line.gate.copy()
        if j is not None: mask[j] = False
        d[mask] = np.clip(d[mask]-deviation[mask], -cfg.max_lfc, cfg.max_lfc)
    factors, projected, mass = project_factors(line.share, d, j, kd)
    # Library size is an independent whole-cell scale after composition projection.
    lib_scale = 1.0
    if cfg.library_size is not None:
        lib_scale = float(cfg.library_size)/max(float(np.median(line.lib[pick])), 1)
    positive = line.share > 0
    effective_pi = min(cfg.pi, float(factors[positive].min())) if positive.any() else 0.0
    esc_rng = np.random.default_rng(seed_for(cfg.seed, line_key, "escapers"))
    escapers = esc_rng.random(len(pick)) < effective_pi
    cols = np.flatnonzero((np.abs(factors*lib_scale-1) > 1e-12) | ((line.share == 0) & (projected > 0)))
    for g in cols:
        # Fresh per-gene seed gives common random draws without target-order dependence.
        grng = np.random.default_rng(seed_for(cfg.seed, line_key, "gene", line.genes[g]))
        factor = factors[g]
        inside = max((factor-effective_pi)/(1-effective_pi), 0)
        F = np.where(escapers, 1.0, inside)*lib_scale
        original = B[:, g].copy()
        values = grng.binomial(original, np.minimum(F, 1.0))
        up = np.maximum(F-1, 0)
        base = line.lib[pick]*line.share[g]
        if line.share[g] == 0 and projected[g] > 0:
            # Induced genes absent from controls must be able to gain counts.
            intensity = line.lib[pick]*projected[g]*lib_scale
        else:
            expected = cfg.up_w*original+(1-cfg.up_w)*base
            intensity = up*expected
        if np.any(intensity > 0):
            dispersion = float(line.overdispersion[g])
            if dispersion > 1e-8:
                intensity = grng.gamma(1/dispersion, intensity*dispersion)
            values += grng.poisson(intensity)
        B[:, g] = values
    if B.max(initial=0) > np.iinfo(np.int32).max: raise ValueError("count exceeds int32 range")
    out = sp.csr_matrix(B.astype(np.int32)); out.eliminate_zeros(); validate_counts(out)
    planned = np.log2((projected*1e6+0.5)/(line.share*1e6+0.5))
    return out, dict(net_share_change=mass-1, projection_normalizer=mass, n_changed=int(len(cols)),
                     effective_pi=effective_pi, library_scale=lib_scale), planned


def target_metadata(path, line, fallback_n, enabled=True):
    """Optional CSV metadata; supports shared or per-context rows/column suffixes."""
    df = pd.read_csv(path)
    if 'target_gene' not in df: raise ValueError("pert_counts.csv needs target_gene")
    if 'context' in df: df = df[df.context.astype(str) == str(line)]
    if df.target_gene.duplicated().any(): raise ValueError("duplicate target rows; supply context labels")
    result = {}
    for row in df.to_dict('records'):
        n, umi = fallback_n, None
        if enabled:
            for field, default in [('n_cells', fallback_n), ('median_umi_per_cell', None)]:
                value = row.get(field+'_'+str(line), row.get(field, default))
                if value is None or pd.isna(value): value = default
                if value is not None and (not np.isfinite(value) or value <= 0): raise ValueError(f"invalid {field} for {row['target_gene']}")
                if field == 'n_cells':
                    if float(value) != int(value): raise ValueError("n_cells must be integer")
                    n = int(value)
                else: umi = None if value is None else float(value)
        result[str(row['target_gene'])] = dict(n_cells=n, median_umi_per_cell=umi)
    if not result: raise ValueError(f"no requested targets for context {line}")
    return result


def check_output(path, expected_shape, block=512):
    a = ad.read_h5ad(path, backed='r')
    try:
        if a.shape != expected_shape: raise ValueError(f"output shape {a.shape} != {expected_shape}")
        stored = 0
        for start in range(0, a.n_obs, block):
            matrix = a.X[start:start+block]; validate_counts(matrix, f"output rows {start}:{start+block}")
            if np.any(np.asarray(matrix.sum(1)).ravel() > 1e6): raise ValueError('cell exceeds 1,000,000 count cap')
            stored += matrix.nnz if sp.issparse(matrix) else matrix.size
        if a.n_obs > 400000 or stored > 4750000000: raise ValueError('submission exceeds cell or stored-entry cap')
    finally: a.file.close()
    return True


def answer_free_check(cells, line, planned, j):
    pooled = np.asarray(cells.sum(0)).ravel(); pooled = pooled/max(pooled.sum(), 1)*1e6
    realized = np.log2((pooled+0.5)/(line.share*1e6+0.5))
    mask = np.isfinite(realized) & (line.share > 0)
    if j is not None: mask[j] = False
    return dict(projected_effect_mae=float(np.abs(realized[mask]-planned[mask]).mean()),
                n_cells=cells.shape[0], median_umi_per_cell=float(np.median(np.asarray(cells.sum(1)).ravel())))


def prediction_ontarget(path, targets):
    p = Path(path)
    if not p.is_dir() or not (p/'ontarget_lfc.npy').exists(): return np.full(len(targets), np.nan)
    names = np.load(p/'targets.npy').astype(str); values = np.load(p/'ontarget_lfc.npy'); pos = dict(zip(names, values))
    return np.asarray([pos.get(t, np.nan) for t in targets])


def cmd_submit(a):
    cfg = Cfg(a)
    if a.settings:
        settings = json.loads(Path(a.settings).read_text())
        if settings.get('schema') != SCHEMA: raise ValueError('use generator settings from this notebook')
        chosen = settings['settings']
        for key in ['alpha', 'z', 'min_effect', 'call', 'top_n', 'sign_rule', 'detect_floor', 'z_req', 'gate_from', 'min_bg',
                    'pi', 'up_w', 'n_cells', 'metadata', 'match_pool', 'tilt']:
            if key in chosen: setattr(cfg, key, chosen[key])
        if a.delta_array == 'selected': a.delta_array = settings['delta_array']
    if a.delta_array == 'selected': raise ValueError('--delta-array selected needs --settings')
    data = Path(a.data); genes = read_list(data/'gene_names.csv', 'gene_name')
    gp = {g: i for i, g in enumerate(genes)}; deltas = dict(x.split('=', 1) for x in a.delta)
    specifications = {line: target_metadata(data/'pert_counts.csv', line, cfg.n_cells, cfg.metadata) for line in a.lines}
    sets = [set(s) for s in specifications.values()]
    if any(s != sets[0] for s in sets): raise ValueError('all submitted contexts must share the exact perturbation panel')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True); path = out/'prediction.h5ad'
    if path.exists(): raise FileExistsError('choose a new output directory to preserve the previous submission')
    for line_specs in specifications.values():
        if any(s['n_cells'] != a.required_n_cells for s in line_specs.values()):
            raise ValueError(f'2026 submissions require exactly {a.required_n_cells} cells per target; optional CSV metadata must respect that contract')
    frames = [pd.DataFrame(dict(target_gene=t, context=line), index=[f'{line}_{t}_{i:05d}' for i in range(spec['n_cells'])])
              for line in a.lines for t, spec in specifications[line].items()]
    obs = pd.concat(frames); obs['target_gene'] = obs.target_gene.astype('category'); obs['context'] = obs.context.astype('category')
    report = dict(settings=cfg.as_dict(), delta_array=a.delta_array, lines={})
    with h5py.File(path, 'w') as f:
        write_elem(f, 'obs', obs); write_elem(f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))
        f.attrs['encoding-type'] = 'anndata'; f.attrs['encoding-version'] = '0.1.0'; ds = None
        for name in a.lines:
            if name not in deltas: raise ValueError(f'no predictions for {name}')
            line = Line.from_h5ad(data/f'context_{name}.h5ad', genes)
            targets = list(specifications[name]); D, info = load_effects(deltas[name], targets, genes, a.delta_array, cfg)
            D = calibrate(D, line, targets, cfg, info=info); del info
            own = prediction_ontarget(deltas[name], targets)
            if a.kd_orion or a.kd_tcell:
                legacy_own = kd_from_sources(targets, genes, a.kd_orion or (), a.kd_tcell, a.kd_tcell_condition)
                own = np.where(np.isfinite(own), own, legacy_own)
            kd = [None] * len(targets) if is_nochange(a.delta_array) else kd_residual(own, cfg); checks = []; diags = []
            # Pick max count once; different n_cells use prefixes of this same pool.
            pool_cfg = copy.copy(cfg); pool_cfg.n_cells = max(s['n_cells'] for s in specifications[name].values())
            common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(cfg.seed, name, 'base_controls')))
            for i, t in enumerate(targets):
                local = copy.copy(cfg); local.n_cells = specifications[name][t]['n_cells']; local.library_size = specifications[name][t]['median_umi_per_cell']
                pick = None if local.tilt else common[:local.n_cells]
                cells, diag, planned = generate_one(line, D[i], gp.get(t), kd[i], local, pick=pick, line_key=name, target_key=t)
                diags.append(dict(target=t, **diag))
                if i < a.checks: checks.append(dict(target=t, **answer_free_check(cells, line, planned, gp.get(t))))
                # float32 holds these integer counts exactly and matches vcc prep's output encoding,
                # so prep packages without a second full-matrix copy (int32 needs ~37 GB at 360k cells).
                if cells.data.max(initial=0) >= 2**24: raise ValueError('count too large for exact float32')
                cells = cells.astype(np.float32)
                if ds is None: write_elem(f, 'X', int64_indptr(cells)); ds = sparse_dataset(f['X'])
                else: ds.append(cells)
            pd.DataFrame(checks).to_csv(out/f'checks_{name}.csv', index=False)
            pd.DataFrame(diags).to_csv(out/f'projection_{name}.csv', index=False)
            report['lines'][name] = dict(n_targets=len(targets), n_cells=sum(s['n_cells'] for s in specifications[name].values()),
                metadata=specifications[name], measured_ontarget=int(np.isfinite(own).sum()))
            print(f"generated context {name}", flush=True)
    report['integer_counts'] = check_output(path, (len(obs), len(genes)))
    save_json(out/'generator_report.json', report); print(f'WROTE {path}; checked every output row')


class StreamingH5AD:
    """An .h5ad whose obs is known up front and whose X arrives one block at a time.

    Nothing but the current block is held in memory. The row pointer is int64 from the first
    block, so the finished file may exceed 2**31 stored values. Every block is cast to the first
    block's dtype, and the row total is checked against obs on close.
    """
    def __init__(self, path, obs, genes):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        self.n = len(obs); self.rows = 0; self.ds = None; self.dtype = None
        self.f = h5py.File(self.path, 'w')
        self.f.attrs['encoding-type'] = 'anndata'; self.f.attrs['encoding-version'] = '0.1.0'
        write_elem(self.f, 'obs', obs); write_elem(self.f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))

    def append(self, block):
        block = block.tocsr() if sp.issparse(block) else sp.csr_matrix(np.asarray(block))
        if self.ds is None:
            self.dtype = block.dtype
            write_elem(self.f, 'X', int64_indptr(block)); self.ds = sparse_dataset(self.f['X'])
        else:
            self.ds.append(block.astype(self.dtype, copy=False))
        self.rows += block.shape[0]

    def close(self):
        self.f.close()
        if self.rows != self.n: raise ValueError(f'{self.path}: wrote {self.rows} rows for {self.n} obs entries')


def labelled_obs(labels, prefix):
    labels = np.asarray(labels, dtype=str)
    return pd.DataFrame({'target_gene': pd.Categorical(labels)}, index=[f'{prefix}{i:07d}' for i in range(len(labels))])


def h1_controls(h, labels, genes, seed):
    """The benchmark's seeded control split: the generator half (a Line) and the scorer's half (rows). The gate and
    the detection floor's reference size come from the scorer's half, which is what the evaluator compares against."""
    ci = np.flatnonzero(labels == CONTROL)
    if len(ci) < 2: raise ValueError('H1 needs controls')
    ci = np.random.default_rng(seed_for(seed, 'H1', 'controls')).permutation(ci)
    train_rows, score_rows = ci[:len(ci) // 2], ci[len(ci) // 2:]
    small = h[np.sort(train_rows)].to_memory(); validate_counts(small.X, 'H1 raw counts')
    line = Line(aligned_X(small, genes), None, genes); del small
    # The scorer uses its REAL control half. Match that exact gate, even when
    # the generator half happens to put a near-threshold gene on the other side.
    line.gate = expression_gate(moments(h, np.sort(score_rows), alignment(h.var_names.astype(str), genes))['mean_cpm'])
    line.n_reference = len(score_rows)
    return line, train_rows, score_rows


def reference_targets(real_path):
    """Targets and their real cell counts in an existing scoring reference (a previous stage's real.h5ad)."""
    a = ad.read_h5ad(real_path, backed='r')
    try: counts = a.obs['target_gene'].astype(str).value_counts()
    finally: a.file.close()
    counts = counts.drop(CONTROL)
    return sorted(counts.index), {t: int(n) for t, n in counts.items()}


def stream_h1(a, genes, real_path, line_pseudo=0.5, block=4096):
    """Write the scoring reference straight from the backed H1 file, one target at a time.

    Controls are split into two disjoint, seeded halves: the first half becomes the generator's
    input (held in memory, it is sampled for every target); the second half is appended to the
    reference after the perturbed cells. In oracle mode each target's cells are split the same
    way as before: the estimation half only produces that target's effect vector, the other half
    is appended to the reference. Peak memory is the control half plus one target.

    Returns (Line, targets, {target: reference cell count}, oracle effects or None, oracle own LFCs).
    """
    h = ad.read_h5ad(a.cells, backed='r')
    try:
        labels = h.obs[a.obs_col].astype(str).values
        requested = sorted(set(read_list(a.targets, 'target_gene'))) if a.targets else sorted(set(labels) - {CONTROL})
        counts = pd.Series(labels).value_counts()
        targets = [t for t in requested if counts.get(t, 0) >= a.min_cells and t != CONTROL]
        if a.n_targets and len(targets) > a.n_targets:
            targets = sorted(np.random.default_rng(seed_for(a.seed, 'H1', 'panel')).choice(targets, a.n_targets, replace=False).tolist())
        if not targets: raise ValueError('no eligible H1 targets')

        def subset(rows):
            rows = np.sort(np.asarray(rows))
            small = h[rows].to_memory(); validate_counts(small.X, 'H1 raw counts')
            return aligned_X(small, genes)

        line, train_rows, score_rows = h1_controls(h, labels, genes, a.seed)
        save_json(Path(real_path).parent / 'control_split.json', dict(seed=a.seed,
                  generator_controls=len(train_rows), scoring_controls=len(score_rows),
                  gate_source='scoring real controls', genes_above_5_cpm=int(line.gate.sum())))
        plan = {}
        for t in targets:
            rows = np.flatnonzero(labels == t)
            if a.mode == 'oracle':
                rows = np.random.default_rng(seed_for(a.seed, 'H1', t, 'split')).permutation(rows)
                plan[t] = (rows[:len(rows) // 2], rows[len(rows) // 2:])
            else:
                plan[t] = (None, rows)
        real_counts = {t: len(ref) for t, (_, ref) in plan.items()}
        obs_labels = [t for t in targets for _ in range(real_counts[t])] + [CONTROL] * len(score_rows)
        writer = StreamingH5AD(real_path, labelled_obs(obs_labels, 'real_'), genes)
        oracle, own = ([], []) if a.mode == 'oracle' else (None, None)
        gp = {g: i for i, g in enumerate(genes)}
        try:
            for i, t in enumerate(targets):
                estimation, reference = plan[t]
                if estimation is not None:
                    x = subset(estimation); lib = np.asarray(x.sum(1)).ravel()
                    mean = np.asarray((sp.diags(1e6 / np.maximum(lib, 1)) @ x).mean(0)).ravel()
                    d = np.log2((mean + line_pseudo) / (line.cpm + line_pseudo)); oracle.append(d)
                    own.append(d[gp[t]] if t in gp else np.nan); del x
                writer.append(subset(reference))
                print(f'  reference: {i + 1}/{len(targets)} targets', end='\r', flush=True)
            sorted_score = np.sort(score_rows)
            for start in range(0, len(sorted_score), block):
                writer.append(subset(sorted_score[start:start + block]))
        finally:
            writer.close()
        print(f'\n  wrote {real_path}: {writer.rows:,} cells ({len(score_rows):,} scoring controls)', flush=True)
        return (line, targets, real_counts, None if oracle is None else np.asarray(oracle),
                None if own is None else np.asarray(own))
    finally:
        h.file.close()


def write_arm(path, line, effects, kd, targets, n_cells, common, cfg, line_key='H1', block=4096):
    """Generate each target's cells and append them immediately, then the generator's control cells.

    cell-eval2 requires the predicted file to carry the control label even though vcc2026 scores
    every perturbation against the REAL controls (`control_source: real`), so the generator's own
    input controls are appended as before; they are not used as the scoring reference.
    """
    gp = {g: i for i, g in enumerate(line.genes)}
    labels = [t for t in targets for _ in range(n_cells[t])] + [CONTROL] * line.n
    writer = StreamingH5AD(path, labelled_obs(labels, 'pred_'), line.genes)
    try:
        for i, t in enumerate(targets):
            local = copy.copy(cfg); local.n_cells = n_cells[t]
            cells = generate_one(line, effects[i], gp.get(t), kd[i], local, pick=common[:local.n_cells],
                                 line_key=line_key, target_key=t)[0]
            writer.append(cells)
        for start in range(0, line.n, block):
            writer.append(line.X[start:start + block].astype(np.int32))
    finally:
        writer.close()


class OfficialEvaluator:
    """Runs official cell-eval2 against one reference file, and scores arms with the leaderboard formula.

    The reference never changes between arms, so its expensive side (real DE tests, pseudobulks, noise corrections)
    is cached once with `prep-cache --side real` and reused by every arm through `--cache-real`. The ceiling and the
    competition bundle (cell-eval2's generic-response baseline as the 0 end, a real-vs-real replicate anchor as the
    1 end) depend on the real data only, so they are built once too, and a later stage can point at them
    (`reference=` a previous benchmark folder) instead of rebuilding them.
    """
    def __init__(self, a, reference, out, reuse=None):
        self.binary = shutil.which(a.cell_eval)
        if self.binary is None: raise RuntimeError('install official cell-eval2 (pip install -r requirements-eval.txt) before running the H1 benchmark')
        help_text = subprocess.run([self.binary, 'run', '--help'], check=True, text=True, capture_output=True).stdout
        if '--ceiling' not in help_text or '--cache-real' not in help_text or '--write-degenes' not in help_text:
            raise RuntimeError('installed cell-eval2 lacks --ceiling/--cache-real/--write-degenes; install requirements-eval.txt')
        self.a, self.reference, self.out = a, Path(reference), Path(out)
        home = Path(reuse) if reuse else self.out
        self.cache, self.bundle = home / 'real_cache', home / 'bundle'
        self.common = ['--pert-col', 'target_gene', '--control', CONTROL, '--profile', a.eval_profile,
                       '--input-type', 'counts', '--preset', 'vcc2026', '--set', 'de.backend=pdex',
                       '--set', f'num_threads={a.num_threads}']
        self.invocations = []

    def _run(self, command):
        print('  $ ' + ' '.join(shlex.quote(str(x)) for x in command), flush=True)
        subprocess.run([str(x) for x in command], check=True); self.invocations.append([str(x) for x in command])

    def prepare(self):
        # One-sided cache warm: the expression comparator must be named. vcc2026 on counts resolves
        # to bulk_lognorm (cell_eval2.competition.competition_comparator()).
        self._run([self.binary, 'prep-cache', '--side', 'real', '--adata', self.reference,
                   '--cache-real', self.cache, '--comparator', 'bulk_lognorm', *self.common])
        self._run([self.binary, 'run', '-ar', self.reference, '-o', self.out / 'ceiling', '--ceiling',
                   '--cache-real', self.cache, *self.common])

    def build_bundle(self):
        """The leaderboard's two ends for this reference: generic baseline (0) and replicate anchor (1)."""
        base = self.out / 'baseline'
        self._run([self.binary, 'baseline', '-ar', self.reference, '-o', base, '--save-pred', base / 'baseline_pred.h5ad',
                   '--cache-real', self.cache, *self.common])
        self._run([self.binary, 'prep-real-bundle', '--real', self.reference, '--baseline', base / 'baseline_pred.h5ad',
                   '-o', self.bundle, '--anchor-splits', str(self.a.anchor_splits), '--cache-real', self.cache, *self.common])
        (base / 'baseline_pred.h5ad').unlink()                  # large; the bundle keeps only aggregates

    def score(self, prediction, output):
        self._run([self.binary, 'run', '-ap', prediction, '-ar', self.reference, '-o', output,
                   '--cache-real', self.cache, '--write-degenes', *self.common])
        self.record(Path(output) / 'invocation.json')

    def scale(self, output):
        """The leaderboard number for one scored arm: cell-eval2 score against the bundle (from_replicate)."""
        output = Path(output)
        self._run([self.binary, 'score', '--user-agg', output / 'agg_results.csv', '--real-bundle', self.bundle,
                   '--user-meta', output / 'run_meta.json', '-o', output / 'scaled.csv'])
        return read_scaled(output / 'scaled.csv')

    def record(self, path):
        import importlib.metadata
        versions = {}
        for package in ['cell-eval2', 'pdex', 'numpy', 'scipy', 'anndata']:
            try: versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError: versions[package] = 'not installed in generator environment'
        save_json(path, dict(commands=self.invocations, versions=versions))


def read_scaled(path):
    """A `cell-eval2 score` CSV -> {metric: leaderboard value}, with 'overall' = avg_score (from_replicate)."""
    frame = pd.read_csv(path).set_index('metric')
    values = frame['from_replicate'].astype(float).to_dict()
    values['overall'] = values.pop('avg_score')
    return values


def per_target_table(arm_dir, alpha=0.05):
    """Per knockdown: our called and the real DE genes (scorer's p_adj < alpha, own gene excluded), their overlap,
    precision against chance (random genes hit real DE genes at n_real / scored genes), direction accuracy, recall."""
    pred = pd.read_parquet(Path(arm_dir) / 'de_pred.parquet'); real = pd.read_parquet(Path(arm_dir) / 'de_real.parquet')
    rows = []
    for t, r in real.groupby('target'):
        p = pred[pred.target == t]
        P = set(p.feature[(p.p_adj < alpha) & (p.feature != t)]); R = set(r.feature[(r.p_adj < alpha) & (r.feature != t)])
        both = P & R; same = 0
        if both:
            lp = p.set_index('feature').log2_fold_change; lr = r.set_index('feature').log2_fold_change
            same = int(sum(np.sign(lp[x]) == np.sign(lr[x]) for x in both))
        scored = r.feature.nunique(); precision = len(both) / len(P) if P else np.nan
        rows.append(dict(target=t, n_called=len(P), n_real=len(R), overlap=len(both), same_direction=same,
                         precision=precision, chance_precision=len(R) / scored if scored else np.nan,
                         enrichment=precision * scored / len(R) if R else np.nan,
                         direction_accuracy=same / len(both) if both else np.nan, recall=len(both) / len(R) if R else np.nan))
    return pd.DataFrame(rows).set_index('target')


def per_target_summary(table):
    return dict(median_n_called=float(table.n_called.median()), median_n_real=float(table.n_real.median()),
                median_precision=float(table.precision.median()), median_chance_precision=float(table.chance_precision.median()),
                median_enrichment=float(table.enrichment.median()), direction_accuracy=float(table.same_direction.sum() / max(table.overlap.sum(), 1)),
                median_recall=float(table.recall.median()))


def metric_rank_table(outputs, directions_path):
    """Rank only explicitly requested metric families, with explicit directions.

    The evaluator emits diagnostics as well as scoring metrics. Never average
    their ranks blindly. Supply metric,direction ('min'/'max') from current rules.
    """
    directions = pd.read_csv(directions_path)
    if not {'metric', 'direction'} <= set(directions) or directions.metric.duplicated().any():
        raise ValueError('directions CSV needs unique metric,direction columns')
    values = []
    for label, path in outputs.items():
        df = pd.read_csv(Path(path)/'agg_results.csv')
        if not {'metric', 'value'} <= set(df):
            if {'metric', 'score'} <= set(df): df = df.rename(columns={'score': 'value'})
            elif set(directions.metric) <= set(df.columns):
                # cell-eval2 writes a wide describe-style frame; use only its mean row.
                mean_row = np.zeros(len(df), bool)
                for col in df.columns:
                    if col not in set(directions.metric): mean_row |= df[col].astype(str).eq('mean').to_numpy()
                if not mean_row.any() and len(df) == 1: mean_row[:] = True
                if mean_row.sum() != 1: raise ValueError('official aggregate CSV needs exactly one mean row')
                df = pd.DataFrame({'metric': directions.metric, 'value': [df.loc[mean_row, m].iloc[0] for m in directions.metric]})
            else: raise ValueError(f'unrecognized official aggregate schema: {df.columns.tolist()}')
        for metric, direction in directions[['metric', 'direction']].itertuples(index=False, name=None):
            v = df.loc[df.metric == metric, 'value']
            if len(v) != 1 or not np.isfinite(v.iloc[0]): raise ValueError(f'{label}: metric {metric} missing/nonfinite')
            if direction not in ['min', 'max']: raise ValueError('metric direction must be min/max')
            values.append(dict(variant=label, metric=metric, value=float(v.iloc[0]), direction=direction))
    frame = pd.DataFrame(values); frame['rank'] = np.nan
    for metric in directions.metric:
        use = frame.metric == metric
        frame.loc[use, 'rank'] = frame.loc[use, 'value'].rank(ascending=frame.loc[use, 'direction'].iloc[0] == 'min', method='average')
    return frame, frame.groupby('variant')['rank'].mean().sort_values(kind='stable')


ARM_KEYS = ['method', 'call', 'top_n', 'sign_rule', 'min_bg', 'detect_floor', 'z_req', 'gate_from', 'alpha', 'z', 'min_effect', 'pi', 'up_w', 'cell_rule']


def benchmark_variants(a):
    """The arms to score: an explicit JSON list (--variants) or the product of the grid flags."""
    if a.variants:
        variants = json.loads(Path(a.variants).read_text())
        for v in variants:
            unknown = set(v) - set(ARM_KEYS) - {'label'}
            if unknown or 'method' not in v: raise ValueError(f'bad variant {v}: needs method, unknown keys {sorted(unknown)}')
            v.setdefault('cell_rule', 'fixed')
        return variants
    return [dict(method=m, call=a.call, top_n=a.top_n, sign_rule=a.sign_rule, detect_floor=a.detect_floor, z_req=a.z_req,
                 alpha=alpha, z=z, min_effect=me, pi=pi, up_w=w, cell_rule=rule)
            for m in a.delta_array for z in a.z_grid for me in a.min_effect_grid for alpha in a.alpha_grid
            for pi in a.pi_grid for w in a.up_w_grid for rule in a.cell_rules]


def cmd_benchmark(a):
    """Official H1 benchmark, streamed: reference and each arm go to disk one target at a time.

    Every arm is scored raw (agg_results.csv), with the leaderboard formula (scaled.csv) and per knockdown
    (per_target.csv). Selection uses the scaled overall. Oracle mode never writes selected generator settings.
    """
    genes = read_list(a.genes, 'gene_name')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    if a.mode == 'transfer':
        provenance = json.loads((Path(a.predictions)/'prediction_report.json').read_text())
        if provenance.get('heldout_background') != 'H1': raise ValueError('generator tuning requires the H1 validation: predictions with H1 held out')
        if provenance.get('schema') != SCHEMA: raise ValueError('rebuild the validation predictions with vcc3_ridge.py')
        if 'H1' in provenance.get('source_backgrounds', []): raise ValueError('H1 leaked into the validation sources')
    if a.mode == 'transfer':
        variants = benchmark_variants(a)
    elif a.variants:
        # Oracle arms on the same real cells: the oracle effects and/or true no change, at any alpha and cell rule.
        variants = benchmark_variants(a)
        if {v['method'] for v in variants} - {'oracle', 'nochange'}: raise ValueError('oracle arms use method oracle or nochange')
    else:
        variants = [dict(method='oracle', alpha=a.alpha, pi=a.pi, up_w=a.up_w, cell_rule='matched')]
    if a.mode == 'oracle' and (a.reference or a.resume):
        raise ValueError('the oracle estimates effects from its own split of the real cells: no --reference or --resume')
    if a.mode == 'transfer':
        unknown = {v['method'] for v in variants} - set(provenance['models']) - {'nochange'}
        if unknown: raise ValueError(f'unknown models {sorted(unknown)}; the exam has {provenance["models"]}')
        # Fail before the (hours-long) reference build: an SE gate needs an SE table, its own or the gating array's.
        for v in variants:
            se_from = v.get('gate_from') or v['method']
            if float(v.get('z', 0) or 0) > 0 and v.get('call', 'z') == 'z' and not (Path(a.predictions) / f'{se_from}_se.npy').exists():
                raise ValueError(f"{v.get('label', v['method'])}: z* > 0 needs {se_from}_se.npy; ridge outputs have none, so use z = 0 "
                                 "or gate_from='re'")
    if a.reference:
        # A previous stage's reference: same real cells, cache and bundle; rebuild only the generator's controls.
        ref = Path(a.reference); real_path = ref / 'real.h5ad'
        if not (ref / 'bundle').exists(): raise ValueError(f'{ref} has no competition bundle')
        targets, real_counts = reference_targets(real_path)
        h = ad.read_h5ad(a.cells, backed='r')
        try: line = h1_controls(h, h.obs[a.obs_col].astype(str).values, genes, a.seed)[0]
        finally: h.file.close()
        oracle_effects = oracle_own = None
        evaluator = OfficialEvaluator(a, real_path, out, reuse=ref)
        save_json(out / 'reference.json', dict(reused=str(ref.resolve()), targets=len(targets)))
    elif a.resume and all((out / x).exists() for x in ('real.h5ad', 'real_cache/manifest.json', 'ceiling/ceiling_agg.csv', 'control_split.json')):
        # An interrupted run of this folder: keep its real cells, cache and ceiling; build only what is missing.
        real_path = out / 'real.h5ad'
        if json.loads((out / 'control_split.json').read_text()).get('seed') != a.seed: raise ValueError('--resume with a different --seed')
        targets, real_counts = reference_targets(real_path)
        h = ad.read_h5ad(a.cells, backed='r')
        try: line = h1_controls(h, h.obs[a.obs_col].astype(str).values, genes, a.seed)[0]
        finally: h.file.close()
        oracle_effects = oracle_own = None
        evaluator = OfficialEvaluator(a, real_path, out)
        if a.mode == 'transfer' and not (out / 'bundle/manifest.json').exists():
            for partial in (out / 'baseline', out / 'bundle'):           # left by the interrupted run
                if partial.exists(): shutil.rmtree(partial)
            evaluator.build_bundle()
        print(f'resumed {out}: {len(targets)} targets, reference reused', flush=True)
    else:
        real_path = out/'real.h5ad'
        line, targets, real_counts, oracle_effects, oracle_own = stream_h1(a, genes, real_path)
        evaluator = OfficialEvaluator(a, real_path, out); evaluator.prepare()
        if a.mode == 'transfer' or a.scale: evaluator.build_bundle()
    print(f'{len(variants)} arms to score on {len(targets)} targets', flush=True)
    if a.mode == 'transfer':
        kd = kd_residual(prediction_ontarget(a.predictions, targets), Cfg(a))
        effects = {}
        for m, gate in sorted({(v['method'], v.get('gate_from', '') or '') for v in variants}):
            effects[m, gate] = ((np.zeros((len(targets), len(genes)), np.float32), {}) if is_nochange(m) else
                                (load_deltas(a.predictions, targets, genes, m), load_info(a.predictions, targets, genes, m, gate)))
    else:
        # Ceiling-only oracle inputs: per-cell arithmetic CPM from each target's estimation half.
        effects = {('oracle', ''): (oracle_effects, {}), ('nochange', ''): (np.zeros_like(oracle_effects), {})}
        kd = kd_residual(oracle_own, Cfg(a))
    outputs, settings, scaled, summaries = {}, {}, {}, {}
    maximum = max([a.n_cells] + list(real_counts.values()))
    pool_cfg = Cfg(a); pool_cfg.n_cells = maximum
    common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(a.seed, 'H1', 'base_controls')))
    for index, variant in enumerate(variants):
        method = variant['method']; effects_array, info = effects[method, variant.get('gate_from', '') or '']
        label = variant.get('label') or f'{method}_{index:03d}'
        cfg = Cfg(argparse.Namespace(**dict(vars(a), **{k: v for k, v in variant.items() if k not in ('method', 'cell_rule', 'label')})))
        calibrated = calibrate(effects_array, line, targets, cfg, info=info)
        n_cells = {t: (real_counts[t] if variant['cell_rule'] == 'matched' else a.n_cells) for t in targets}
        pred_path = out/f'{label}.h5ad'
        write_arm(pred_path, line, calibrated, [None] * len(targets) if is_nochange(method) else kd, targets, n_cells, common, cfg)
        del calibrated
        evaluator.score(pred_path, out/label)
        if not a.keep_arm_files: pred_path.unlink()   # ~2 GB each; the scores are in out/label
        outputs[label] = str(out/label); settings[label] = dict(variant)
        save_json(out/f'{label}.json', settings[label])
        table = per_target_table(out/label); table.to_csv(out/label/'per_target.csv')
        summaries[label] = per_target_summary(table)
        if a.mode == 'transfer' or a.scale: scaled[label] = evaluator.scale(out/label)
        print(f'official evaluation complete: {label}' + (f" (scaled overall {scaled[label]['overall']:.4f})" if label in scaled else ''), flush=True)
    pd.DataFrame(summaries).T.to_csv(out/'per_target_summary.csv')
    save_json(out/'benchmark.json', dict(mode=a.mode, outputs=outputs, settings=settings, n_targets=len(targets),
              ceiling=str(evaluator.out / 'ceiling') if not a.reference else str(Path(a.reference) / 'ceiling'),
              real_cache=str(evaluator.cache), bundle=str(evaluator.bundle),
              note='H1 is generator calibration; final evidence requires unseen lines. Oracle is a generator-only ceiling check.'))
    if a.metric_directions or a.eval_profile == 'vcc2026':
        directions = a.metric_directions or str(out/'metric_directions.csv')
        if a.metric_directions is None:
            pd.DataFrame({'metric': list(VCC2026_METRICS), 'direction': list(VCC2026_METRICS.values())}).to_csv(directions, index=False)
        frame, ranks = metric_rank_table(outputs, directions)
        frame.to_csv(out/'metric_ranks.csv', index=False); ranks.to_csv(out/'aggregate_rank.csv')
    if a.mode == 'oracle' and scaled:
        table = pd.DataFrame(scaled).T.sort_values('overall', ascending=False)
        table.index.name = 'arm'; table.to_csv(out/'scaled_scores.csv')
    if a.mode == 'transfer':
        table = pd.DataFrame(scaled).T.sort_values('overall', ascending=False)
        table.index.name = 'arm'; table.to_csv(out/'scaled_scores.csv')
        eligible = [label for label in table.index if settings[label]['cell_rule'] == 'fixed']
        if not eligible: raise ValueError('include fixed cell rule to select submission-compatible generator settings')
        best = settings[eligible[0]]
        chosen = {k: v for k, v in best.items() if k not in ('method', 'cell_rule', 'label')}
        margin = {}
        if a.margin_over:
            if a.margin_over not in table.index: raise ValueError(f'--margin-over {a.margin_over!r} is not an arm here')
            margin = dict(margin_over=a.margin_over, margin_over_overall=float(table.loc[a.margin_over, 'overall']),
                          margin=float(table.loc[eligible[0], 'overall'] - table.loc[a.margin_over, 'overall']))
        save_json(out/'selected_generator.json', dict(settings=dict(Cfg(argparse.Namespace(**dict(vars(a), **chosen))).as_dict() | chosen, n_cells=a.n_cells),
                   cell_rule='fixed', n_cells=a.n_cells, delta_array=best['method'], arm=eligible[0],
                   scaled_overall=float(table.loc[eligible[0], 'overall']), **margin,
                   selection='scaled overall (leaderboard formula)', schema=SCHEMA,
                   note='H1 is calibration, not an untouched test'))


def add_switches(p):
    p.add_argument('--alpha', type=float, default=1.0); p.add_argument('--n-cells', type=int, default=400)
    p.add_argument('--pi', type=float, default=0); p.add_argument('--up-w', type=float, default=0.75)
    p.add_argument('--metadata', action=argparse.BooleanOptionalAction, default=True)
    p.add_argument('--z', type=float, default=0); p.add_argument('--min-effect', type=float, default=0)
    p.add_argument('--call', choices=['z', 'topn'], default='z'); p.add_argument('--top-n', type=int, default=1000)
    p.add_argument('--sign-rule', action='store_true'); p.add_argument('--detect-floor', action='store_true')
    p.add_argument('--z-req', type=float, default=4.0); p.add_argument('--gate-from', default='')
    p.add_argument('--min-bg', type=int, default=1)
    p.add_argument('--match-pool', action='store_true'); p.add_argument('--tilt', action='store_true')
    p.add_argument('--seed', type=int, default=2026)


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__); sub = p.add_subparsers(dest='cmd', required=True)
    s = sub.add_parser('submit'); add_switches(s)
    for key in ['data', 'out']: s.add_argument('--'+key, required=True)
    s.add_argument('--lines', nargs='+', default=['A', 'B', 'C']); s.add_argument('--delta', action='append', required=True)
    s.add_argument('--required-n-cells', type=int, default=400)
    s.add_argument('--delta-array', default='selected'); s.add_argument('--settings')
    s.add_argument('--kd-orion', nargs='*'); s.add_argument('--kd-tcell'); s.add_argument('--kd-tcell-condition', default='Stim48hr')
    s.add_argument('--checks', type=int, default=20)
    for command, mode in [('benchmark', 'transfer'), ('oracle', 'oracle')]:
        b = sub.add_parser(command); add_switches(b); b.set_defaults(mode=mode)
        for key in ['cells', 'genes', 'out']: b.add_argument('--'+key, required=True)
        b.add_argument('--predictions', required=mode == 'transfer'); b.add_argument('--targets')
        b.add_argument('--obs-col', default='target_gene'); b.add_argument('--delta-array', nargs='+', default=['re'])
        b.add_argument('--min-cells', type=int, default=100); b.add_argument('--n-targets', type=int, default=0)
        b.add_argument('--alpha-grid', nargs='+', type=float, default=[0.75, 1.0, 1.5, 2.0])
        b.add_argument('--z-grid', nargs='+', type=float, default=[2.0, 3.0, 4.0])
        b.add_argument('--min-effect-grid', nargs='+', type=float, default=[0.1, 0.2])
        b.add_argument('--pi-grid', nargs='+', type=float, default=[0.0])
        b.add_argument('--up-w-grid', nargs='+', type=float, default=[0.75])
        b.add_argument('--cell-rules', nargs='+', choices=['matched', 'fixed'], default=['fixed'])
        b.add_argument('--cell-eval', default='cell-eval2'); b.add_argument('--eval-profile', default='vcc2026')
        b.add_argument('--num-threads', type=int, default=4)
        b.add_argument('--keep-arm-files', action='store_true', help='keep each scored arm .h5ad (deleted by default)')
        b.add_argument('--variants', help='JSON list of arms (method, call, top_n, sign_rule, detect_floor, z_req, alpha, z, min_effect, ...)')
        b.add_argument('--reference', help='a previous benchmark folder: reuse its real.h5ad, real_cache, ceiling and bundle')
        b.add_argument('--anchor-splits', type=int, default=5, help='replicate-anchor splits for the competition bundle')
        b.add_argument('--margin-over', help="an arm label; selected_generator.json records the winner's scaled-overall margin over it")
        b.add_argument('--scale', action='store_true', help='oracle: also build the competition bundle on its real cells and scale every arm')
        b.add_argument('--resume', action='store_true', help="reuse this folder's real cells, cache, ceiling (and bundle, if complete) from an interrupted run")
        b.add_argument('--metric-directions', help='CSV metric,direction for current official scoring metrics only')
    a = p.parse_args()
    if a.cmd == 'submit': cmd_submit(a)
    else: cmd_benchmark(a)

### vcc3_screen.py
Effect-space screen: every array × α × minimum size on all H1 validation targets, scored against H1's measured effects without generating cells.

In [ ]:
%%writefile ~/vcc/code/vcc_2026_3c/vcc3_screen.py
"""Screen H1 effect arrays without generating cells, using the generator's exact calibration.

Each target contributes equally. Differences and paired 95% bootstrap CIs are
arm minus re (alpha=1, min_effect=0); lower MSE and higher other metrics are better.
Only finite measured effects on H1 ctrl_cpm > 5 genes are scored; own genes are
excluded. Constant-vector Pearson is defined as zero.

  top50_overlap       share of the 50 strongest NONZERO predictions that are among the 50 strongest measured
                      genes (denominator 50): an all-zero prediction scores 0, never a chance fill by gene order
  direction_accuracy  sign agreement on significant genes (|lfc| / SE > 3) that the arm actually predicts (nonzero)
  direction_coverage  share of the significant genes the arm predicts at all; accuracy and coverage are kept apart
  MSE is reported, not used for selection: H1's measured effects are mostly noise, so squared error rewards
  shrinking toward zero. Pearson, top-50 overlap and direction accuracy do not depend on alpha, so the screen
  chooses the array and minimum size; alpha is left to the official benchmark.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
import re
from types import SimpleNamespace

import anndata as ad
import h5py
import numpy as np
import pandas as pd
try:
    from anndata.io import read_elem
except ImportError:  # older anndata
    from anndata.experimental import read_elem

from vcc2_common import expression_gate, read_names, save_json
from vcc3_generate import Cfg, calibrate, load_deltas

ALPHAS = (0.5, 0.75, 1.0, 1.5, 2.0)
MINIMUMS = (0.0, 0.1, 0.2)
METRICS = ('mse', 'pearson', 'top50_overlap', 'direction_accuracy', 'direction_coverage')


def unique_names(values, label):
    names = pd.Index(np.asarray(values, dtype=str))
    if not names.is_unique: raise ValueError(f'{label}: duplicate names')
    return names


def required_indices(source, requested, label):
    idx = source.get_indexer(requested)
    if (idx < 0).any():
        raise ValueError(f'{label}: missing {np.asarray(requested)[idx < 0].tolist()}')
    return idx


def load_inputs(predictions, truth, filters, targets_csv):
    p = Path(predictions)
    report = json.loads((p / 'prediction_report.json').read_text())
    if report.get('heldout_background') != 'H1' or 'H1' in report.get('source_backgrounds', []):
        raise ValueError('screen requires predictions fitted with H1 held out')
    genes = unique_names(np.load(p / 'genes.npy'), 'prediction genes')
    pred_targets = unique_names(np.load(p / 'targets.npy'), 'prediction targets')
    requested = unique_names(read_names(targets_csv, 'target_gene'), 'validation targets')
    required_indices(pred_targets, requested, 'prediction targets')
    panel = ad.read_h5ad(truth)
    ti = required_indices(unique_names(panel.obs_names, 'truth targets'), requested, 'truth targets')
    gi = required_indices(unique_names(panel.var_names, 'truth genes'), genes, 'truth genes')
    kept = panel.obs.iloc[ti]['status'].astype(str).to_numpy() == 'kept'
    targets = requested[kept].to_numpy(dtype=str); ti = ti[kept]
    if not len(targets): raise ValueError('no kept validation targets')
    measured = np.asarray(panel.layers['lfc'])[np.ix_(ti, gi)].astype(np.float64)
    se = np.asarray(panel.layers['se'])[np.ix_(ti, gi)].astype(np.float64)
    with h5py.File(filters, 'r') as f: var = read_elem(f['var'])
    unique_names(var.index, 'filter genes')
    cpm = var['ctrl_cpm'].reindex(genes).to_numpy(float)
    gate = expression_gate(cpm)
    eligible = np.isfinite(measured) & gate[None, :] & (targets[:, None] != genes.to_numpy()[None, :])
    if (eligible.sum(1) == 0).any(): raise ValueError('validation target has no eligible measured genes')
    significant = eligible & np.isfinite(se) & (se >= 0) & (np.abs(measured) > 3 * se)
    info = dict(requested_targets=len(requested), n_targets=len(targets),
                excluded_not_kept=requested[~kept].tolist(), genes=len(genes), gated_genes=int(gate.sum()),
                genes_absent_from_filter=int((var.index.get_indexer(genes) < 0).sum()),
                n_significant_targets=int(significant.any(1).sum()),
                measured_genes_min=int(eligible.sum(1).min()), measured_genes_max=int(eligible.sum(1).max()))
    return genes.to_numpy(dtype=str), targets, measured, se, eligible, significant, gate, info


def target_metrics(predicted, measured, eligible, significant):
    out = np.full((len(measured), len(METRICS)), np.nan)
    for i in range(len(measured)):
        x, y = predicted[i, eligible[i]].astype(float), measured[i, eligible[i]]
        out[i, 0] = np.mean((x - y) ** 2)
        xc, yc = x - x.mean(), y - y.mean()
        denom = np.linalg.norm(xc) * np.linalg.norm(yc)
        out[i, 1] = np.dot(xc, yc) / denom if denom > 0 else 0.0
        n = min(50, len(x))
        nonzero = np.flatnonzero(x != 0)
        top_x = nonzero[np.argsort(-np.abs(x[nonzero]), kind='stable')[:n]]       # only genes the arm predicts
        top_y = np.argsort(-np.abs(y), kind='stable')[:n]
        out[i, 2] = np.isin(top_x, top_y).sum() / n
        sig = significant[i]
        if sig.any():
            called = sig & (predicted[i] != 0)
            out[i, 4] = called.sum() / sig.sum()
            if called.any(): out[i, 3] = np.mean(np.sign(predicted[i, called]) == np.sign(measured[i, called]))
    return out


def paired_summary(values, reference, draws):
    diff = values - reference
    with np.errstate(invalid='ignore', divide='ignore'):
        sampled = diff[draws]
        valid = np.isfinite(sampled)
        boot = np.divide(np.where(valid, sampled, 0).sum(1), valid.sum(1),
                         out=np.full((len(draws), len(METRICS)), np.nan), where=valid.sum(1) > 0)
    out = {}
    for j, metric in enumerate(METRICS):
        finite = np.isfinite(values[:, j]); b = boot[np.isfinite(boot[:, j]), j]
        out[metric] = float(values[finite, j].mean()) if finite.any() else np.nan
        out[metric + '_n_targets'] = int(finite.sum())
        out[metric + '_diff'] = float(diff[finite, j].mean()) if finite.any() else np.nan
        lo, hi = np.percentile(b, [2.5, 97.5]) if len(b) else (np.nan, np.nan)
        out[metric + '_diff_ci_low'], out[metric + '_diff_ci_high'] = float(lo), float(hi)
    return out


def shortlist(frame):
    """Best array and minimum size per family by Pearson at alpha = 1 (Pearson does not depend on alpha), ties by
    top-50 overlap; plus any arm whose top-50 overlap is clearly above Model 2's."""
    chosen = []
    at_one = frame[frame.alpha == 1.0]
    for family in ('re', 'programs', 'corrected'):
        candidates = at_one[at_one.family == family].sort_values(['pearson', 'top50_overlap', 'arm'], ascending=[False, False, True])
        if candidates.empty: raise ValueError(f'no candidates for {family}')
        row = candidates.iloc[0]
        chosen.append(dict(arm=row.arm, delta_array=row.delta_array, alpha=float(row.alpha),
                           min_effect=float(row.min_effect), reason=f'best {family} Pearson'))
    winners = at_one[(at_one.family != 'zero') & (at_one.top50_overlap_diff_ci_low > 0)]
    if not winners.empty:
        row = winners.sort_values(['top50_overlap', 'mse', 'arm'], ascending=[False, True, True]).iloc[0]
        if row.arm not in {x['arm'] for x in chosen}:
            chosen.append(dict(arm=row.arm, delta_array=row.delta_array, alpha=float(row.alpha),
                               min_effect=float(row.min_effect), reason='best top-50 overlap with CI above re'))
    return chosen


def screen(predictions, truth, filters, targets_csv, output, seed=2026, resamples=1000):
    if resamples < 1: raise ValueError('resamples must be positive')
    genes, targets, measured, se, eligible, significant, gate, info = load_inputs(predictions, truth, filters, targets_csv)
    p = Path(predictions); out = Path(output); out.mkdir(parents=True, exist_ok=True)
    models = sorted(f.stem for f in p.glob('*.npy') if re.fullmatch(r're|zero|(?:programs|corrected)_k\d+', f.stem))
    if not {'re', 'zero'}.issubset(models): raise ValueError('screen requires re.npy and zero.npy')
    line = SimpleNamespace(genes=genes, gate=gate)
    values, settings = {}, []
    expected_shape = (len(np.load(p / 'targets.npy')), len(genes))
    for model in models:
        raw = np.load(p / f'{model}.npy', mmap_mode='r')
        if raw.shape != expected_shape: raise ValueError(f'{model}: wrong array shape {raw.shape}')
        if model == 'zero' and not np.all(raw == 0): raise ValueError('zero.npy is not all zero')
        D = load_deltas(p, targets, genes, model)  # generator convention: all non-finite predictions -> 0
        for alpha in ALPHAS:
            for minimum in MINIMUMS:
                arm = f'{model}_a{alpha:g}_m{minimum:g}'
                cfg = Cfg(SimpleNamespace(alpha=alpha, min_effect=minimum, z=0))
                transformed = calibrate(D, line, targets, cfg)
                values[arm] = target_metrics(transformed, measured, eligible, significant)
                settings.append(dict(arm=arm, delta_array=model, family=model.split('_k')[0], alpha=alpha, min_effect=minimum))
        print(f'screen: {model}, {len(targets)} targets, 15 settings', flush=True)
    draws = np.random.default_rng(seed).integers(0, len(targets), size=(resamples, len(targets)))
    np.save(out / 'bootstrap_indices.npy', draws)
    pd.DataFrame(dict(target_gene=targets, n_genes=eligible.sum(1), n_significant_genes=significant.sum(1))).to_csv(out / 'targets.csv', index=False)
    reference = values['re_a1_m0']
    rows, per_target = [], []
    zero = values['zero_a1_m0']
    for setting in settings:
        metrics = values[setting['arm']]
        rows.append(dict(**setting, **paired_summary(metrics, reference, draws)))
        t = pd.DataFrame(metrics, columns=METRICS)
        t.insert(0, 'target_gene', targets); t.insert(0, 'arm', setting['arm']); per_target.append(t)
    frame = pd.DataFrame(rows).sort_values(['mse', 'arm']).reset_index(drop=True)
    frame.to_csv(out / 'screen.csv', index=False)
    pd.concat(per_target, ignore_index=True).to_csv(out / 'per_target.csv', index=False)
    finalists = shortlist(frame)
    # Alignment gate: zero must be worst on the signal metrics (Pearson and top-50 overlap). Every shortlisted arm
    # must beat it with a paired 95% CI above zero on both; MSE is not a gate (it favours shrinking to zero).
    zero_comparisons = []
    for setting in settings:
        if setting['family'] == 'zero': continue
        comp = paired_summary(values[setting['arm']], zero, draws)
        zero_comparisons.append(dict(arm=setting['arm'], mse_diff=comp['mse_diff'],
                                     pearson_diff=comp['pearson_diff'], pearson_ci_low=comp['pearson_diff_ci_low'],
                                     top50_diff=comp['top50_overlap_diff'], top50_ci_low=comp['top50_overlap_diff_ci_low']))
    shortlisted = {x['arm'] for x in finalists}
    worse = [x for x in zero_comparisons if x['pearson_diff'] <= 0 or x['top50_diff'] <= 0]
    unclear = [x for x in zero_comparisons if x['arm'] in shortlisted and (x['pearson_ci_low'] <= 0 or x['top50_ci_low'] <= 0)]
    passed = not unclear
    save_json(out / 'zero_comparisons.json', zero_comparisons)
    save_json(out / 'shortlist.json', dict(arms=finalists, ready_for_benchmark=passed))
    info.update(seed=seed, bootstrap_resamples=resamples, bootstrap_reference='re_a1_m0',
                bootstrap_indices_sha256=hashlib.sha256(draws.tobytes()).hexdigest(),
                inputs=dict(predictions=str(p.resolve()), truth=str(Path(truth).resolve()),
                            filters=str(Path(filters).resolve()), targets=str(Path(targets_csv).resolve())),
                n_arms=len(frame), zero_mse=float(zero[:, 0].mean()), zero_check_passed=passed,
                arms_not_better_than_zero=len(worse), shortlisted_not_clearly_better_than_zero=len(unclear),
                zero_check='Every shortlisted arm must beat zero on Pearson and top-50 overlap with paired 95% CIs above 0; MSE is reported, not gated.',
                transform='vcc3_generate.load_deltas + calibrate: minimum size, alpha, +/-8, ctrl_cpm > 5, own gene zero; nonfinite predictions -> 0',
                scoring='Gate genes with finite truth, excluding own gene; equal target weight; no clipping of measured truth.',
                pearson_constant=0, top50='nonzero predictions only, ties in stable gene order; denominator min(50, eligible genes)',
                direction='accuracy on significant genes the arm predicts; coverage = share of significant genes predicted',
                selection='array and minimum size by Pearson at alpha = 1; alpha is chosen by the official benchmark')
    save_json(out / 'screen_report.json', info)
    cols = ['arm', *METRICS, 'pearson_diff_ci_low', 'pearson_diff_ci_high', 'top50_overlap_diff_ci_low']
    ordered = frame.sort_values(['pearson', 'arm'], ascending=[False, True])
    summary = ordered[cols].to_string(index=False, float_format=lambda x: f'{x:.6f}')
    summary += '\n\nShortlist:\n' + json.dumps(finalists, indent=2)
    summary += (f'\n\nZero gate: {"PASS" if passed else "FAIL"}; {len(worse)} arms are not above zero on Pearson and top-50, '
                f'{len(unclear)} shortlisted arms are not clearly above it.\n')
    (out / 'summary.txt').write_text(summary)
    print(summary, flush=True)
    return info


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    for arg in ('predictions', 'truth', 'filters', 'targets', 'out'): parser.add_argument('--' + arg, required=True)
    parser.add_argument('--seed', type=int, default=2026)
    parser.add_argument('--resamples', type=int, default=1000)
    args = parser.parse_args()
    report = screen(args.predictions, args.truth, args.filters, args.targets, args.out, args.seed, args.resamples)
    if not report['zero_check_passed']:
        raise SystemExit('STOP: a shortlisted arm does not clearly beat zero on Pearson and top-50. Inspect screen_report.json and alignment before benchmarking.')

## 2. What calling costs on the profile metrics (item 0, part 1)
Every `vcc_2026_3` finalist arm, from its saved scores (nothing rerun): raw and scaled pds and mse, genes called per knockdown, and the generic baseline's raw values (the 0 of the leaderboard scale). mse is `expr_mse_unbiased_capped_norm`, lower is better; an arm scores above 0 only if its raw value is below the baseline's.

In [ ]:
final = W3 / 'val_final'
raw = pd.read_csv(final / 'metric_ranks.csv').pivot(index='variant', columns='metric', values='value')
scaled = pd.read_csv(final / 'scaled_scores.csv', index_col='arm')
profile = pd.DataFrame({'pds_raw': raw['pds_cosine'], 'pds_scaled': scaled['pds_cosine'],
                        'mse_raw': raw['expr_mse_unbiased_capped_norm'], 'mse_scaled': scaled['expr_mse_unbiased_capped_norm'],
                        'overall': scaled['overall'],
                        'genes_called': pd.read_csv(final / 'per_target_summary.csv', index_col=0)['median_n_called']})
baseline = pd.read_csv(final / 'bundle/baseline_agg.csv', index_col='statistic').loc['mean']
display(profile.sort_values('pds_raw', ascending=False).round(4))
print(f"generic baseline (the 0): pds {baseline['pds_cosine']:.4f}, mse {baseline['expr_mse_unbiased_capped_norm']:.4f}")

Read on 2026-10-15, before any new run:
- **Calling does not cost pds; it helps.** Top 1,000 / 2,000 + floor score raw pds 0.822 / 0.819 against 0.813 for every gene at α = 2 and 0.803 for the gate.
- **mse is 0 for every arm, and not by a small margin.** The bar is the generic baseline's 0.953; true no change scores 1.147, the closest arm (programs, α = 1) 1.115, top 2,000 + floor 2.34 and every gene at α = 2 6.89. Larger predicted effects make mse worse, so the arms below ask how much smaller effects help (item A) and whether even the true effects can get under the bar (item 0b).

## 3. Honest-profile arms (item A)
Model 2 with every gene at its own Δ: no top-N, no detection floor, no minimum size, at α ∈ {0.5, 0.75, 1} (`vcc_2026_3`'s non-gated arm used α = 2). Scored against `vcc_2026_3`'s finalists bundle (`--reference`), so it compares directly with every earlier arm. About 30 min per arm: 1.5 hours. Needs the memory that generation and `vcc prep` use.

In [ ]:
honest = [dict(method='re', call='z', z=0.0, min_effect=0.0, alpha=alpha, label=f'model2_all_alpha{alpha:g}')
          for alpha in (0.5, 0.75, 1.0)]
variants = Path(ENV['WORK']) / 'val_3c_variants.json'; variants.write_text(json.dumps(honest, indent=1))
display(pd.DataFrame(honest))
bg('val_3c', f'"$PY" vcc3_generate.py benchmark --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK3/validation/targets.csv" '
             f'--predictions "$WORK3/validation/pred/H1" --eval-profile vcc2026 --n-targets 75 --reference "$WORK3/val_final" '
             f'--variants {shlex.quote(str(variants))} --out "$WORK/val_3c"')

Progress, then the honest-profile arms next to the finalists that bracket them (non-gated α = 2, top 2,000 + floor, no change), with the paired bootstrap against submission 3's recipe (as in `vcc_2026_3b`).

In [ ]:
def arms_table(folder, labels=None):
    scaled = pd.read_csv(folder / 'scaled_scores.csv', index_col='arm')
    if labels: scaled = scaled.loc[labels]
    settings = pd.DataFrame({label: json.loads((folder / f'{label}.json').read_text()) for label in scaled.index}).T
    per_target = pd.read_csv(folder / 'per_target_summary.csv', index_col=0)
    raw = pd.read_csv(folder / 'metric_ranks.csv').pivot(index='variant', columns='metric', values='value').add_prefix('raw_')
    return settings.join(scaled.add_prefix('scaled_')).join(per_target).join(raw)
SUBMITTED = 'model2_top2000_floor'
status('val_3c', n=5)
if (Path(ENV['WORK']) / 'val_3c/scaled_scores.csv').exists():
    table = pd.concat([arms_table(Path(ENV['WORK']) / 'val_3c'),
                       arms_table(W3 / 'val_final', ['model2_nongated', SUBMITTED, 'true_nochange'])])
    display(table.sort_values('scaled_overall', ascending=False))

In [ ]:
BOOT_METRICS = {'pds_cosine': 1, 'de_wilcoxon_direction_fidelity_yield_raw': 1, 'de_wilcoxon_direction_reach_raw': 1,
                'de_wilcoxon_sig_jaccard': 1, 'de_wilcoxon_lfc_nmae': -1}                  # the 6th, mse, is 0 for every arm
def bootstrap(folders, reference=SUBMITTED, resamples=5000, seed=2026):
    bundle = W3 / 'val_final/bundle'
    b = pd.read_csv(bundle / 'baseline_agg.csv', index_col='statistic').loc['mean', list(BOOT_METRICS)].to_numpy(float)
    a = pd.read_parquet(bundle / 'anchor_agg.parquet').set_index('metric').loc[list(BOOT_METRICS), 'replicate'].to_numpy(float)
    arms = {p.name: p for folder in folders for p in sorted(folder.iterdir()) if (p / 'results.csv').exists() and (p / 'scaled.csv').exists()}
    per = {}
    for name, p in arms.items():
        r = pd.read_csv(p / 'results.csv')
        per[name] = r[r.metric.isin(BOOT_METRICS)].pivot(index='perturbation', columns='metric', values='value')
    targets = sorted(set.intersection(*(set(t.index) for t in per.values())))
    X = np.stack([per[n].loc[targets, list(BOOT_METRICS)].to_numpy(float) for n in arms])     # arms x targets x metrics
    overall = lambda Xs: ((np.nanmean(Xs, axis=-2) - b) / (a - b)).sum(-1) / 6               # nmae is NaN on 5 targets
    point = overall(X)
    official = np.array([pd.read_csv(p / 'scaled.csv').set_index('metric').loc['avg_score', 'from_replicate'] for p in arms.values()])
    if not np.allclose(point, official, atol=1e-6): raise ValueError('the rebuilt overall does not match the official one')
    rng = np.random.default_rng(seed)
    boot = np.stack([overall(X[:, rng.integers(0, len(targets), len(targets)), :]) for _ in range(resamples)])
    k = list(arms).index(reference); d = boot - boot[:, [k]]
    return pd.DataFrame(dict(overall=point, diff=point - point[k], ci_low=np.quantile(d, 0.025, axis=0),
                             ci_high=np.quantile(d, 0.975, axis=0), share_ahead=(d > 0).mean(0)), index=list(arms)).sort_values('overall', ascending=False)
if (Path(ENV['WORK']) / 'val_3c/scaled_scores.csv').exists():
    display(bootstrap([W3 / 'val_final', W3B / 'val_3b', Path(ENV['WORK']) / 'val_3c']).round(4))

### Results (recorded 2026-10-15)
`val_3c` finished 02:06 UTC. H1 overall on `vcc_2026_3`'s finalists bundle; "genes DE" is the median number of genes per knockdown that come out significant in our generated cells (scorer's test), not a call list:

| Arm | overall | vs submission 3 | 95% interval | ahead in | nmae | fid | reach | jac | pds | raw mse | genes DE | precision | direction |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **every gene, α = 0.75** | **0.132** | **+0.014** | **+0.002 to +0.026** | **99%** | −0.316 | 0.259 | 0.007 | 0.202 | 0.637 | 1.85 | 1,909 | 0.227 | 0.608 |
| every gene, α = 1 | 0.131 | +0.014 | −0.003 to +0.030 | 95% | −0.491 | 0.355 | 0.007 | 0.274 | 0.644 | 2.50 | 2,998 | 0.204 | 0.600 |
| top 2,000 + floor (submission 3) | 0.118 | — | — | — | −0.389 | 0.270 | 0.005 | 0.164 | 0.657 | 2.34 | 1,995 | 0.190 | 0.629 |
| every gene, α = 0.5 | 0.109 | −0.009 | −0.024 to +0.006 | 12% | −0.178 | 0.116 | 0.006 | 0.097 | 0.612 | 1.39 | 805 | 0.257 | 0.617 |
| every gene, α = 2 (`model2_nongated`) | 0.015 | −0.103 | −0.150 to −0.060 | 0% | −1.393 | 0.436 | 0.011 | 0.393 | 0.644 | 6.89 | 5,545 | 0.185 | 0.579 |

- **Every gene at 0.75 × its own Δ is the first arm to beat submission 3's recipe beyond the noise** (+0.014; the whole 95% interval above 0). α = 1 is as good on average but less certain.
- **The gain is not on the profile metrics.** mse is still 0 (raw 1.85 against the bar of 0.953) and pds is slightly lower (0.637 vs 0.657). It comes from the DE metrics: Jaccard 0.202 vs 0.164 and nmae −0.32 vs −0.39 at the same fidelity.
- **Without any calling, about as many genes come out DE as top 2,000 calls** (1,909 vs 1,995 at α = 0.75), and more of them are real DE genes (precision 0.227 vs 0.190), at slightly worse direction (0.608 vs 0.629). Letting the generator's own noise decide which predicted changes show up selects better than ranking by |Δ| / SE and raising the winners to the detection floor.
- **α sets the number of DE genes and trades nmae against fidelity and Jaccard**, as N did for top-N: 805 genes at 0.5, 1,909 at 0.75, 2,998 at 1, 5,545 at 2; 0.75–1 is the optimum on H1.
- **Caveats for the leaderboard:** the gain is mostly Jaccard and nmae, the two metrics H1 judged differently from A–C (H1 rewarded extra calls on Jaccard and penalised the floor on nmae more than the leaderboard did), so the leaderboard gain may be smaller than +0.014.

## 4. H1 oracle on the leaderboard scale (item 0, part 2)
Each of the 75 H1 knockdowns' cells is split in two: one half estimates that knockdown's true effect, the other half is the real side it is scored against (`vcc_2026_3`'s `oracle` mode). With `--scale` the oracle builds its own generic baseline and replicate anchor on that real side, so every arm gets leaderboard-scale scores. Arms on the same real cells:
- **oracle, matched cell counts** — the classic ceiling;
- **oracle, 400 cells per knockdown** — as our submissions generate;
- **oracle × 0.5, 400 cells** — mse rewards shrinking noisy effects; can a smaller true effect get under the bar?
- **no change, 400 cells** — the floor on these real cells.

The real side, its cache and the bundle take about 4 hours at 75 targets, then about 30 min per arm. It waits for item A's run to finish (memory).

In [ ]:
oracle_arms = [dict(method='oracle', alpha=1.0, cell_rule='matched', label='oracle_matched'),
               dict(method='oracle', alpha=1.0, cell_rule='fixed', label='oracle_400'),
               dict(method='oracle', alpha=0.5, cell_rule='fixed', label='oracle_400_alpha0.5'),
               dict(method='nochange', alpha=1.0, cell_rule='fixed', label='nochange_400')]
oracle_variants = Path(ENV['WORK']) / 'oracle_variants.json'; oracle_variants.write_text(json.dumps(oracle_arms, indent=1))
display(pd.DataFrame(oracle_arms))
bg('oracle', 'while tmux has-session -t vcc3c_val_3c 2>/dev/null; do sleep 60; done\n'
             f'"$PY" vcc3_generate.py oracle --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK3/validation/targets.csv" '
             f'--eval-profile vcc2026 --n-targets 75 --anchor-splits 5 --scale --variants {shlex.quote(str(oracle_variants))} --out "$WORK/oracle_h1"')

Progress, then every oracle arm on the leaderboard scale and raw, with this real side's generic baseline (the 0) and replicate anchor (the 1). The decisive line: does the oracle score mse above 0?

In [ ]:
status('oracle', n=5)
oracle_dir = Path(ENV['WORK']) / 'oracle_h1'
if (oracle_dir / 'scaled_scores.csv').exists():
    o_scaled = pd.read_csv(oracle_dir / 'scaled_scores.csv', index_col='arm')
    o_raw = pd.read_csv(oracle_dir / 'metric_ranks.csv').pivot(index='variant', columns='metric', values='value')
    display(o_scaled.round(4)); display(o_raw[o_scaled.columns.drop('overall')].round(4))
    ends = pd.DataFrame({'generic baseline (0)': pd.read_csv(oracle_dir / 'bundle/baseline_agg.csv', index_col='statistic').loc['mean'],
                         'replicate (1)': pd.read_parquet(oracle_dir / 'bundle/anchor_agg.parquet').set_index('metric')['replicate']})
    display(ends.loc[o_scaled.columns.drop('overall')].round(4))
    print('oracle mse above 0 (scaled):', {arm: round(float(v), 4) for arm, v in o_scaled['expr_mse_unbiased_capped_norm'].items()})

### Results (recorded 2026-10-15)
`oracle_h1` finished 07:40 UTC (75 targets; real side = the half of each knockdown's cells not used to estimate its effect, with its own generic baseline and replicate anchor):

| Arm | overall | mse | pds | fid | reach | jac | nmae | raw mse |
|---|---|---|---|---|---|---|---|---|
| oracle, matched cell counts | **0.914** | 0.815 | 1.035 | 0.918 | 1.004 | 0.699 | 1.016 | 0.181 |
| oracle, 400 cells (as submitted) | **0.854** | **0.768** | 1.021 | 0.692 | 0.996 | 0.662 | 0.986 | 0.225 |
| oracle × 0.5, 400 cells | 0.602 | 0.633 | 1.003 | 0.133 | 0.867 | 0.394 | 0.582 | 0.353 |
| no change, 400 cells | −0.059 | 0 | −0.003 | −0.239 | −0.016 | −0.028 | −0.065 | 1.138 |

Bar for mse (generic baseline): raw 0.953; replicate 0.005.

- **The generator is not the bottleneck.** Fed true effects, it scores 0.85 overall at 400 cells and mse 0.77 (raw 0.225, far under the 0.953 bar). The earlier explanation, that our cells are too dispersed for mse, was wrong.
- **mse is 0 because of our effects, not the cells.** No change scores raw 1.14 because real knockdowns do move expression; the generic baseline scores 0.953 because it knows the line's average knockdown response. Every predicted Δ we have scored is further from the truth than no change (honest profile α = 0.75: 1.85; top 2,000 + floor: 2.34), so beating the bar needs effects closer to the real ones than the line's average response — most directly, getting that shared response itself.
- **Shrinking true effects hurts every metric** (× 0.5: mse 0.63, fidelity 0.13), so shrinkage is not a route to mse on its own.
- **400 cells cost the oracle 0.06** against matched counts (mostly fidelity), a ceiling effect of the submission format, not something to tune.

**Implication:** the remaining gains are in the effect model, and the largest unexploited piece is the shared knockdown response that the leaderboard's 0 already contains (plan item 4, the generic-response probe).

## 5. Generate — every gene at α = 0.75
Submission 4: section 3's `model2_all_alpha0.75` (H1 0.132, +0.014 over submission 3's recipe, 95% interval +0.002 to +0.026) with `vcc_2026_3`'s final Δ tables for A–C (Model 2, `re`). α = 1 scored the same on H1 (0.131) but its interval includes 0, so 0.75 is chosen. Needs ~20 GB free disk; about 50 minutes; refuses to overwrite a finished run.

In [ ]:
ARM = 'model2_all_alpha0.75'
SETTINGS = Path(ENV['WORK']) / 'val_3c/selected_generator.json'
chosen = json.loads(SETTINGS.read_text())
if chosen['arm'] != ARM: raise ValueError(f"val_3c selected {chosen['arm']}; write {ARM}'s settings before generating")
s = chosen['settings']
if (chosen['delta_array'], s['call'], s['z'], s['min_effect'], s['detect_floor'], s['alpha']) != ('re', 'z', 0.0, 0.0, False, 0.75):
    raise ValueError(f'unexpected settings for {ARM}: {s}')
LINES = ['A', 'B', 'C']                    # use D/E/F with the final bundle
RUN_NAME = 'vcc_2026_3c_all_alpha075'
result = V / 'results' / RUN_NAME
if result.exists() and any(result.iterdir()): raise FileExistsError('Choose a new RUN_NAME to preserve existing results.')
free = shutil.disk_usage(V).free / 1e9
if free < 20: raise RuntimeError(f'only {free:.0f} GB free; a submission needs ~20 GB')
line_args = ' '.join(shlex.quote(line) for line in LINES)
delta_args = ' '.join(f'--delta {shlex.quote(line + "=" + str(W3 / "pred" / line))}' for line in LINES)
bg('generate', f'"$PY" vcc3_generate.py submit --data "$DATA" --lines {line_args} --delta-array selected --settings {shlex.quote(str(SETTINGS))} {delta_args} --out {shlex.quote(str(result))}')

In [ ]:
status('generate')

## 6. Package
`vcc prep` validates and packages the run. It holds the whole matrix in memory (~30 GB at peak), so it stops early unless RAM + swap is enough.

In [ ]:
SUBMISSION = V / 'results' / RUN_NAME
memory = {k: int(v.split()[0]) / 1e6 for k, v in (line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()) if k in ('MemAvailable', 'SwapFree')}
if memory['MemAvailable'] + memory['SwapFree'] < 34:
    raise RuntimeError(f"only {memory['MemAvailable']:.0f} GB RAM + {memory['SwapFree']:.0f} GB swap free; vcc prep peaks near 30 GB. Add swap first.")
run(f'vcc prep {shlex.quote(str(SUBMISSION / "prediction.h5ad"))} -g "$DATA/gene_names.csv" --perts "$DATA/pert_counts.csv" -o {shlex.quote(str(SUBMISSION / "prediction.vcc"))}')

In [ ]:
SUBMIT_TO_LEADERBOARD = True
MODEL_NAME = 'LHY'   # shown publicly on the leaderboard: keep it free of method details
if SUBMIT_TO_LEADERBOARD:
    print(f"submitting as {MODEL_NAME!r}; private record: {chosen['arm']} ({chosen['delta_array']}), alpha={s['alpha']}, no call, no floor, "
          f"validation scaled overall={chosen['scaled_overall']:.4f}")
    run(f'vcc submit {shlex.quote(str(SUBMISSION / "prediction.vcc"))} -m {shlex.quote(MODEL_NAME)} --wait')

### Leaderboard result (recorded 2026-10-15)
Submission 4 (entry `B2PPQypk9SGBYGdrlZrV`, rank 820): **overall 0.070**, below submission 3's 0.091, although H1 forecast +0.014.

| | pds | mse | nmae | fid | reach | jac | overall |
|---|---|---|---|---|---|---|---|
| v3: top 2,000 + floor | 0.416 | 0 | 0.048 | −0.088 | 0.192 | −0.024 | 0.091 |
| **v4: every gene, α = 0.75** | 0.403 | 0 | 0.119 | −0.241 | 0.149 | −0.011 | **0.070** |
| change | −0.013 | — | +0.071 | **−0.153** | −0.043 | +0.013 | −0.021 |
| H1's forecast of the change | −0.020 | — | +0.073 | −0.010 | +0.002 | +0.038 | +0.014 |

- nmae, Jaccard and pds moved as H1 forecast; **fidelity is where H1 failed**: on A–C the genes that come out DE without calling are wrong in direction far more often, and top-N + floor protects fidelity there in a way H1 does not show (reach fell too).
- The first time H1 ranked two models the wrong way round. H1 remains right on number of calls (1,000 vs 2,000) and calling vs gating, but **not on calling scheme**; a change that trades fidelity for Jaccard / nmae needs a leaderboard check.
- mse stayed 0, as the oracle predicted: the profile metrics need effects closer to the truth, not a different finishing step.

In [ ]:
BACK_UP_TO_BUCKET = False
if BACK_UP_TO_BUCKET:
    run(r"""
    gcloud storage rsync -r "$WORK" "$B/lib/vcc_2026_3c" --exclude='.*\.h5ad$'
    gcloud storage rsync -r "$CODE" "$B/code/vcc_2026_3c"
    gcloud storage cp ~/vcc/vcc_2026_3c.ipynb "$B/code/vcc_2026_3c.ipynb"
    """)